# Transformer-Based Multimodal Trajectory Prediction on the Waymo Open Motion Dataset

A transformer that predicts a road agent's next 8 seconds of motion from 1.1 seconds of history, its neighbors, the local road graph and traffic-light states. It outputs 32 candidate futures ("modes"), each tied to a maneuver (stationary/reverse, straight, left turn, right turn, U-turn), with a confidence and per-timestep uncertainty, and scores the best 6.

## Approach

- **Data:** 600 shards of the Waymo Open Motion Dataset, split into 480 / 60 / 60 shards, i.e. ~1.05M / 131k / 131k samples.
- **Inputs:** target history plus its 7 nearest neighbors, a 768-point road-graph crop centered 30 m ahead of the agent, and up to 16 traffic lights with their time in state. Everything is in the target's agent frame. Maneuver-grouped anchors, i.e each ground-truth future is labeled with a maneuver group from its geometry (heading change, lateral swing, path length). Anchors are fit by k-means within each group: stationary-reverse / straight / left / right / U-turn.
- **Model (6.4M parameters):** a 6-layer Pre-LN transformer encoder over agent and map tokens, then a 6-layer decoder with 32 mode queries. Each query is conditioned on an anchor trajectory and a learned maneuver-group embedding, and predicts cumulative offsets from its anchor, a per-step sigma and a confidence.
- **Loss:** the ground-truth maneuver group is fixed. Within that group, the slot whose predicted path is closest to the truth gets the Gaussian NLL. The confidence target is zero outside the group and a softmax over in-group path errors inside it, so regression and confidence favour the same slot.
- **Evaluation:** top-6 minADE / minFDE / miss rate on the held-out test split, per agent type, plus Waymo-style metrics (3 / 5 / 8 s, speed-scaled lateral / longitudinal miss boxes), calibration and off-road rate.
- **Training:** AdamW with warmup and cosine decay, bf16, 10-hour budget on one A100.


In [ ]:
!pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

tf.config.set_visible_devices([], "GPU")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TensorFlow {tf.__version__}, PyTorch {torch.__version__}, device: {DEVICE}")

In [ ]:
import os, torch

print(f"CPU cores available: {os.cpu_count()}")
print(f"torch threads before: {torch.get_num_threads()}")
torch.set_num_threads(os.cpu_count())
print(f"torch threads after:  {torch.get_num_threads()}")

In [ ]:
# Authenticate gcloud for the Waymo bucket and mount Google Drive.
!gcloud auth login
from google.colab import drive
drive.mount("/content/drive")

# The prepped-data flow keeps no raw shards; the viz cells skip when these lists are empty.
DATA_PATHS = train_paths = val_paths = test_paths = []
MAX_PLAUSIBLE_COORD_M = 400.0   # viz-only distance filter

In [ ]:
# Constants, feature layout, road / traffic-light type buckets and tf.Example parsing.

MAX_AGENTS = 8                # agents kept per sample, including target
PAST_STEPS = 10               # state/past/* steps (1s before current, 0.1s resolution)
HIST_STEPS = PAST_STEPS + 1   # past + current (t=0): what the model sees
FUTURE_STEPS = 80             # 8s of future at 0.1s resolution

MAX_ROAD_POINTS = 768         # map-point budget per sample
ROAD_RADIUS_M = 120.0         # crop radius around the (forward-shifted) crop center
ROAD_POINT_STRIDE = 6         # keep every 6th sample along each polyline
ROAD_FORWARD_OFFSET_M = 30.0  # crop center this far ahead of the agent along its yaw

MAX_TRAFFIC_LIGHTS = 16       # Waymo stores up to 16 lights per scene
MAX_ROAD_TOKENS = MAX_ROAD_POINTS + MAX_TRAFFIC_LIGHTS   # map rows first, then light rows

MAX_TARGETS_PER_SCENE = 2     # targets per scene for viz; data prep keeps all

POS_SCALE_M = 50.0            # input positions divided by this (inputs only; outputs stay in meters)
AGENT_FEAT_DIM = 9            # x, y, vx, vy, cos(dyaw), sin(dyaw), is_vehicle, is_pedestrian, is_cyclist
ROAD_FEAT_DIM = 5             # x, y, dir_x, dir_y, tl_time_in_state (0 for map points)

# Waymo object type codes
TYPE_NAMES = {0: "unset", 1: "vehicle", 2: "pedestrian", 3: "cyclist", 4: "other"}
TYPE_COLORS = {0: "gray", 1: "steelblue", 2: "darkorange", 3: "seagreen", 4: "gray"}
AGENT_TYPE_SLOT = {1: 0, 2: 1, 3: 2}   # one-hot slot within the agent feature vector

# Waymo roadgraph type codes -> buckets (1-3 lanes, 6-13 lines, 15-16 edges, 17-20 point features)
ROAD_TYPE_BUCKET = {t: 1 for t in (1, 2, 3)}
ROAD_TYPE_BUCKET.update({t: 2 for t in range(6, 14)})
ROAD_TYPE_BUCKET.update({t: 3 for t in (15, 16)})
ROAD_TYPE_BUCKET[17] = 4  # stop sign
ROAD_TYPE_BUCKET[18] = 5  # crosswalk
ROAD_TYPE_BUCKET[19] = 6  # speed bump
ROAD_TYPE_BUCKET[20] = 7  # driveway

# Traffic-light bucket = TL_BUCKET_OFFSET + Waymo signal state (0-8)
TL_BUCKET_OFFSET = 8
NUM_TL_STATES = 9
NUM_ROAD_BUCKETS = TL_BUCKET_OFFSET + NUM_TL_STATES   # 17; bucket 0 = unset/other

ROAD_TYPE_LUT = np.zeros(64, dtype=np.int64)   # vectorized type -> bucket lookup
for _t, _b in ROAD_TYPE_BUCKET.items():
    ROAD_TYPE_LUT[_t] = _b

# Plot styles per road bucket and light state
ROAD_BUCKET_STYLE = {
    1: dict(color="lightgray", kind="line", lw=1.0, label="lane center"),
    2: dict(color="khaki", kind="line", lw=1.0, label="road line"),
    3: dict(color="dimgray", kind="line", lw=1.4, label="road edge"),
    4: dict(color="red", kind="point", marker="x", size=40, label="stop sign"),
    5: dict(color="royalblue", kind="point", marker="s", size=10, label="crosswalk"),
    6: dict(color="orange", kind="point", marker="^", size=14, label="speed bump"),
    7: dict(color="silver", kind="line", lw=0.8, label="driveway"),
    0: dict(color="whitesmoke", kind="point", marker=".", size=3, label="other map"),
}
_TL_STYLE = {  # state code -> (color, marker, label)
    0: ("gray", "o", "light: unknown"),
    1: ("red", "D", "light: arrow stop"), 2: ("orange", "D", "light: arrow caution"), 3: ("limegreen", "D", "light: arrow go"),
    4: ("red", "o", "light: stop"), 5: ("orange", "o", "light: caution"), 6: ("limegreen", "o", "light: go"),
    7: ("red", "h", "light: flashing stop"), 8: ("orange", "h", "light: flashing caution"),
}
for _s, (_c, _mk, _lb) in _TL_STYLE.items():
    ROAD_BUCKET_STYLE[TL_BUCKET_OFFSET + _s] = dict(color=_c, kind="point", marker=_mk, size=70, label=_lb)

feature_spec = {
    "state/type": tf.io.FixedLenFeature([128], tf.float32, default_value=None),
    "state/past/x": tf.io.FixedLenFeature([128, PAST_STEPS], tf.float32, default_value=None),
    "state/past/y": tf.io.FixedLenFeature([128, PAST_STEPS], tf.float32, default_value=None),
    "state/past/velocity_x": tf.io.FixedLenFeature([128, PAST_STEPS], tf.float32, default_value=None),
    "state/past/velocity_y": tf.io.FixedLenFeature([128, PAST_STEPS], tf.float32, default_value=None),
    "state/past/bbox_yaw": tf.io.FixedLenFeature([128, PAST_STEPS], tf.float32, default_value=None),
    "state/past/valid": tf.io.FixedLenFeature([128, PAST_STEPS], tf.int64, default_value=None),
    "state/current/x": tf.io.FixedLenFeature([128, 1], tf.float32, default_value=None),
    "state/current/y": tf.io.FixedLenFeature([128, 1], tf.float32, default_value=None),
    "state/current/velocity_x": tf.io.FixedLenFeature([128, 1], tf.float32, default_value=None),
    "state/current/velocity_y": tf.io.FixedLenFeature([128, 1], tf.float32, default_value=None),
    "state/current/bbox_yaw": tf.io.FixedLenFeature([128, 1], tf.float32, default_value=None),
    "state/current/valid": tf.io.FixedLenFeature([128, 1], tf.int64, default_value=None),
    "state/future/x": tf.io.FixedLenFeature([128, FUTURE_STEPS], tf.float32, default_value=None),
    "state/future/y": tf.io.FixedLenFeature([128, FUTURE_STEPS], tf.float32, default_value=None),
    "state/future/valid": tf.io.FixedLenFeature([128, FUTURE_STEPS], tf.int64, default_value=None),
    "state/tracks_to_predict": tf.io.FixedLenFeature([128], tf.int64, default_value=None),
    "roadgraph_samples/xyz": tf.io.FixedLenFeature([30000, 3], tf.float32, default_value=None),
    "roadgraph_samples/dir": tf.io.FixedLenFeature([30000, 3], tf.float32, default_value=None),
    "roadgraph_samples/type": tf.io.FixedLenFeature([30000, 1], tf.int64, default_value=None),
    "roadgraph_samples/id": tf.io.FixedLenFeature([30000, 1], tf.int64, default_value=None),
    "roadgraph_samples/valid": tf.io.FixedLenFeature([30000, 1], tf.int64, default_value=None),
}
# Traffic lights: [time, light] grids (10 past steps + current)
for _k, _dt in (("state", tf.int64), ("valid", tf.int64), ("x", tf.float32), ("y", tf.float32)):
    feature_spec[f"traffic_light_state/past/{_k}"] = tf.io.FixedLenFeature([PAST_STEPS, MAX_TRAFFIC_LIGHTS], _dt, default_value=None)
    feature_spec[f"traffic_light_state/current/{_k}"] = tf.io.FixedLenFeature([1, MAX_TRAFFIC_LIGHTS], _dt, default_value=None)

def parse_scene(serialized_example):
    return tf.io.parse_single_example(serialized_example, feature_spec)

def load_dataset(paths):
    """Loads one shard path or a list of shard paths as a single parsed dataset."""
    if isinstance(paths, str):
        paths = [paths]
    return tf.data.TFRecordDataset(paths, compression_type="").map(parse_scene)

In [ ]:
# Scene extraction: road-graph crop, traffic lights and one sample per tracks_to_predict agent.

def extract_road_graph(parsed_scene, origin_xy, radius=ROAD_RADIUS_M, max_points=MAX_ROAD_POINTS,
                       square=False, forward_dir=None, forward_offset=0.0, stride=1):
    """Crops the road graph. Returns (xy, bucket, feature_id, dir_xy) in world coordinates.

    stride: keep every `stride`-th point per polyline (plus each polyline's
      first point and all stop signs / speed bumps).
    forward_dir / forward_offset: shift the crop center ahead of the agent.
    square=True: square crop (visualization only).
    Beyond max_points, the points farthest from the crop center are dropped.
    """
    xyz = parsed_scene["roadgraph_samples/xyz"].numpy()
    rdir = parsed_scene["roadgraph_samples/dir"].numpy()
    rtype = parsed_scene["roadgraph_samples/type"].numpy().reshape(-1).astype(int)
    rid = parsed_scene["roadgraph_samples/id"].numpy().reshape(-1).astype(int)
    rvalid = parsed_scene["roadgraph_samples/valid"].numpy().reshape(-1).astype(bool)

    empty = (np.zeros((0, 2), np.float32), np.zeros((0,), np.int64),
             np.zeros((0,), np.int64), np.zeros((0, 2), np.float32))

    n = len(rtype)
    if stride > 1:
        starts = np.r_[True, rid[1:] != rid[:-1]]
        seg_start = np.maximum.accumulate(np.where(starts, np.arange(n), 0))
        pos_in_seg = np.arange(n) - seg_start
        point_like = np.isin(rtype, (17, 19))
        keep = rvalid & ((pos_in_seg % stride == 0) | point_like)
    else:
        keep = rvalid
    idx = np.where(keep)[0]
    if len(idx) == 0:
        return empty

    xy, types, ids, dirs = xyz[idx, :2], rtype[idx], rid[idx], rdir[idx, :2]

    center = np.asarray(origin_xy, dtype=np.float64)
    if forward_dir is not None and forward_offset != 0.0:
        center = center + forward_offset * np.asarray(forward_dir, dtype=np.float64)
    offset = xy - center
    dist = np.abs(offset).max(axis=1) if square else np.linalg.norm(offset, axis=1)

    within = np.where(dist <= radius)[0]
    if len(within) > max_points:
        within = within[np.argsort(dist[within])[:max_points]]
        within.sort()  # restore sequential-along-polyline order
    if len(within) == 0:
        return empty

    bucket = ROAD_TYPE_LUT[np.clip(types[within], 0, len(ROAD_TYPE_LUT) - 1)]
    return (xy[within].astype(np.float32), bucket.astype(np.int64),
            ids[within].astype(np.int64), dirs[within].astype(np.float32))


def extract_traffic_lights(parsed_scene):
    """Lights valid at t=0 -> (xy [K, 2] world, bucket [K], time_in_state [K]).
    time_in_state is the fraction of the 1.1s history the light has held its
    current state (1.0 = unchanged, small = just changed)."""
    st = np.concatenate([parsed_scene["traffic_light_state/past/state"].numpy(),
                         parsed_scene["traffic_light_state/current/state"].numpy()], axis=0)   # [11, L]
    va = np.concatenate([parsed_scene["traffic_light_state/past/valid"].numpy(),
                         parsed_scene["traffic_light_state/current/valid"].numpy()], axis=0) > 0
    cx = parsed_scene["traffic_light_state/current/x"].numpy()[0]
    cy = parsed_scene["traffic_light_state/current/y"].numpy()[0]

    lights = np.where(va[-1])[0]
    if len(lights) == 0:
        return np.zeros((0, 2), np.float32), np.zeros((0,), np.int64), np.zeros((0,), np.float32)

    cur = st[-1, lights]
    same = va[:, lights] & (st[:, lights] == cur[None, :])            # [11, K]
    # length of the unbroken run of `same` ending at t=0
    broken = ~same[::-1]                                              # reversed time
    run = np.where(broken.any(axis=0), broken.argmax(axis=0), HIST_STEPS)
    time_in_state = (run / HIST_STEPS).astype(np.float32)

    bucket = TL_BUCKET_OFFSET + np.clip(cur, 0, NUM_TL_STATES - 1)
    xy = np.stack([cx[lights], cy[lights]], axis=1).astype(np.float32)
    return xy, bucket.astype(np.int64), time_in_state


def _hist(parsed_scene, key):
    """[128, HIST_STEPS]: the 10 past steps followed by current (last column = t=0)."""
    past = parsed_scene[f"state/past/{key}"].numpy()
    cur = parsed_scene[f"state/current/{key}"].numpy()
    return np.concatenate([past, cur], axis=1)


def extract_scene_agents(parsed_scene, max_agents=MAX_AGENTS, max_targets=MAX_TARGETS_PER_SCENE):
    x, y = _hist(parsed_scene, "x"), _hist(parsed_scene, "y")
    vx, vy = _hist(parsed_scene, "velocity_x"), _hist(parsed_scene, "velocity_y")
    yaw = _hist(parsed_scene, "bbox_yaw")
    valid = _hist(parsed_scene, "valid")
    obj_type = parsed_scene["state/type"].numpy().astype(int)
    fut_x = parsed_scene["state/future/x"].numpy()
    fut_y = parsed_scene["state/future/y"].numpy()
    fut_valid = parsed_scene["state/future/valid"].numpy()
    to_predict = parsed_scene["state/tracks_to_predict"].numpy()
    tl_xy, tl_type, tl_time = extract_traffic_lights(parsed_scene)

    # each agent's last valid history position, used to rank neighbors
    last_t = np.where(valid > 0, np.arange(HIST_STEPS), -1).max(axis=1)
    has_hist = last_t >= 0
    rows = np.arange(len(last_t))
    safe_t = np.maximum(last_t, 0)
    last_xy = np.stack([x[rows, safe_t], y[rows, safe_t]], axis=1)

    samples = []
    for target_idx in np.where(to_predict == 1)[0]:
        if len(samples) >= max_targets:
            break
        # target needs a valid current state and some future
        if valid[target_idx, -1] == 0 or fut_valid[target_idx].sum() == 0:
            continue

        origin = np.array([x[target_idx, -1], y[target_idx, -1]], dtype=np.float32)
        yaw0 = float(yaw[target_idx, -1])

        others = np.where(has_hist)[0]
        others = others[others != target_idx]
        dists = np.linalg.norm(last_xy[others] - origin, axis=1)
        nearest = others[np.argsort(dists)[: max_agents - 1]]
        selected = np.concatenate([[target_idx], nearest]).astype(int)

        road_xy, road_type, road_id, road_dir = extract_road_graph(
            parsed_scene, origin,
            forward_dir=(math.cos(yaw0), math.sin(yaw0)), forward_offset=ROAD_FORWARD_OFFSET_M,
            stride=ROAD_POINT_STRIDE,
        )

        samples.append({
            "target_idx": 0,               # target is always row 0
            "agent_ids": selected,
            "type": obj_type[selected],
            "x": x[selected], "y": y[selected],                # [A, HIST_STEPS], world coords
            "vx": vx[selected], "vy": vy[selected],
            "bbox_yaw": yaw[selected],
            "valid": valid[selected],
            "future_x": fut_x[selected], "future_y": fut_y[selected],
            "future_valid": fut_valid[selected],
            "origin": origin,              # target's current position (world)
            "yaw0": yaw0,                  # target's current yaw (world) -> agent frame +x
            "road_xy": road_xy, "road_type": road_type,        # world coords, cropped
            "road_id": road_id, "road_dir": road_dir,
            "tl_xy": tl_xy, "tl_type": tl_type, "tl_time": tl_time,   # world coords
        })
    return samples


def draw_road_graph(ax, road_xy, road_type, road_id, zorder=1):
    """Draws polylines (lane centers, road lines, road edges) as lines grouped
    by feature id, and point features and traffic lights as markers."""
    if road_xy is None or len(road_xy) == 0:
        return
    seen_labels = set()
    for bucket, style in ROAD_BUCKET_STYLE.items():
        mask = road_type == bucket
        if mask.sum() == 0:
            continue
        label = style["label"] if style["label"] not in seen_labels else None
        seen_labels.add(style["label"])

        if style["kind"] == "line":
            for feat_id in np.unique(road_id[mask]):
                seg_mask = mask & (road_id == feat_id)
                if seg_mask.sum() < 2:
                    continue
                ax.plot(
                    road_xy[seg_mask, 0], road_xy[seg_mask, 1],
                    "-", color=style["color"], linewidth=style["lw"],
                    alpha=0.9, zorder=zorder, label=label,
                )
                label = None
        else:
            is_light = bucket >= TL_BUCKET_OFFSET
            ax.scatter(
                road_xy[mask, 0], road_xy[mask, 1],
                s=style["size"], c=style["color"], marker=style["marker"],
                alpha=0.9, zorder=zorder + (3 if is_light else 0), label=label,
                edgecolors="black" if is_light else "none", linewidths=0.6,
            )

In [ ]:
# Sanity plot of a few raw scenes in world coordinates.

def plot_scene(scene, title="Scene"):
    fig, ax = plt.subplots(figsize=(6, 6))

    draw_road_graph(ax, scene.get("road_xy"), scene.get("road_type"), scene.get("road_id"))

    seen_labels = set()
    for i in range(scene["x"].shape[0]):
        mask = scene["valid"][i] > 0
        if mask.sum() == 0:
            continue
        xs, ys = scene["x"][i][mask], scene["y"][i][mask]
        obj_type = int(scene["type"][i])
        color = TYPE_COLORS.get(obj_type, "gray")
        type_label = TYPE_NAMES.get(obj_type, "unset")

        if i == 0:
            label = "target" if "target" not in seen_labels else None
            ax.plot(xs, ys, "o-", color="crimson", linewidth=2.5, zorder=3, label=label)
            seen_labels.add("target")
        else:
            label = f"{type_label} (context)" if type_label not in seen_labels else None
            ax.plot(xs, ys, "o-", color=color, alpha=0.6, linewidth=1, zorder=2, label=label)
            seen_labels.add(type_label)

    # target's current yaw defines the agent frame
    ox, oy = scene["origin"]
    ax.annotate("", xy=(ox + 10 * math.cos(scene["yaw0"]), oy + 10 * math.sin(scene["yaw0"])),
                xytext=(ox, oy), arrowprops=dict(arrowstyle="->", color="black", lw=2))

    ax.set_title(title)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.legend(loc="best", fontsize=7)
    ax.set_aspect("equal")
    plt.show()

ds = load_dataset(DATA_PATHS)
for i, parsed_scene in enumerate(ds.take(5)):
    samples = extract_scene_agents(parsed_scene)
    if samples:
        plot_scene(samples[0], title=f"Scene {i} ({len(samples)} target(s))")

In [ ]:
# Model: transformer encoder over agent and map tokens, anchor-conditioned mode decoder, per-mode path, sigma and confidence.

NUM_MODES = 32         # one mode per anchor
EVAL_TOP_K = 6         # we score the best 6 modes (Waymo's convention)
D_MODEL = 192
N_HEADS = 4
N_LAYERS = 6           # encoder layers
DECODER_ROUNDS = 6     # decoder layers (modes attend to each other and to the scene)
FF_DIM = 768
DECODER_FF_DIM = 768   # decoder FFN width and per-timestep MLP width
DROPOUT = 0.1
CUMULATIVE_OFFSETS = True
CONF_HEAD = "mlp"      # "mlp" or "linear"
GROUP_QUERY_EMBED = True   # add a maneuver-group embedding to each mode query

LOG_STD_MIN = -1.6     # clamp the predicted uncertainty to this range
LOG_STD_MAX = 2.0      # (roughly sigma in [0.2, 7.4] metres)


class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""

    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: [seq_len, batch, d_model]
        return x + self.pe[: x.size(0)].unsqueeze(1)


class TrajectoryTransformer(nn.Module):
    def __init__(
        self,
        feat_dim=AGENT_FEAT_DIM,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        decoder_rounds=DECODER_ROUNDS,
        ff_dim=FF_DIM,
        decoder_ff_dim=DECODER_FF_DIM,
        num_modes=NUM_MODES,
        max_agents=MAX_AGENTS,
        past_steps=HIST_STEPS,
        future_steps=FUTURE_STEPS,
        num_road_buckets=NUM_ROAD_BUCKETS,
        dropout=DROPOUT,
        anchor_paths=None,   # [num_modes, future_steps, 2] fixed anchor paths, agent frame
        cumulative_offsets=CUMULATIVE_OFFSETS,
        conf_head_type=CONF_HEAD,
        anchor_group=None,   # [num_modes] long, maneuver group of each slot (grouped anchors only)
        num_groups=0,        # > 0 adds a learned group embedding to each mode query
    ):
        super().__init__()
        self.max_agents = max_agents
        self.past_steps = past_steps
        self.future_steps = future_steps
        self.num_modes = num_modes
        self.cumulative_offsets = cumulative_offsets
        self.conf_head_type = conf_head_type

        # Agent branch
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.agent_embed = nn.Embedding(max_agents, d_model)  # marks which agent is which (target vs neighbors)
        self.pos_encoding = PositionalEncoding(d_model, max_len=past_steps)

        # Road graph + traffic-light branch
        self.road_proj = nn.Linear(ROAD_FEAT_DIM, d_model)
        self.road_type_embed = nn.Embedding(num_road_buckets, d_model)

        # Modality tag: 0 = agent history, 1 = road / traffic light
        self.modality_embed = nn.Embedding(2, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim, dropout=dropout,
            batch_first=False, norm_first=True,  # Pre-LN
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Mode queries conditioned on anchor endpoint (and maneuver group); small init lets the anchor lead
        self.mode_queries = nn.Parameter(torch.randn(num_modes, d_model) * 0.02)
        self.anchor_proj = nn.Linear(2, d_model)  # anchor endpoint (agent frame, /POS_SCALE_M) -> query space
        self.num_groups = num_groups
        if num_groups > 0:
            self.group_embed = nn.Embedding(num_groups, d_model)
            nn.init.normal_(self.group_embed.weight, std=0.02)     # small, like the mode queries
            if anchor_group is None:
                anchor_group = torch.zeros(num_modes, dtype=torch.long)
            self.register_buffer("anchor_group", torch.as_tensor(anchor_group).long().clone())
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=decoder_ff_dim, dropout=dropout,
            batch_first=False, norm_first=True,  # Pre-LN
        )
        self.mode_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_rounds)

        # Per-timestep decode: future positional encoding + shared MLP -> (dx, dy, log_std)
        self.future_pos_encoding = PositionalEncoding(d_model, max_len=future_steps)
        self.decoder_mlp = nn.Sequential(
            nn.Linear(d_model, decoder_ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(decoder_ff_dim, 3),  # (dx, dy, log_std)
        )
        if cumulative_offsets:
            with torch.no_grad():   # start exactly on the anchors
                self.decoder_mlp[-1].weight[:2].zero_()
                self.decoder_mlp[-1].bias[:2].zero_()

        if conf_head_type == "mlp":
            self.conf_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                           nn.ReLU(), nn.Linear(d_model, 1))
        elif conf_head_type == "linear":
            self.conf_head = nn.Linear(d_model, 1)
        else:
            raise ValueError(f"Unknown conf_head_type: {conf_head_type}")

        if anchor_paths is None:
            anchor_paths = torch.zeros(num_modes, future_steps, 2)
        self.register_buffer("anchor_paths", anchor_paths)

    def set_anchors(self, anchor_paths):
        """Overwrite the fixed anchor trajectories in place."""
        self.anchor_paths.copy_(anchor_paths.to(self.anchor_paths.device))

    def _rotated_anchors(self, heading):
        """Rotate the anchor bank [M, T, 2] by each sample's heading [B, 2] -> [B, M, T, 2].
        Identity for agent-frame inputs (heading = [1, 0])."""
        theta = torch.atan2(heading[:, 1], heading[:, 0])  # [B]
        cos_t, sin_t = torch.cos(theta), torch.sin(theta)  # [B]

        anchors = self.anchor_paths.unsqueeze(0)            # [1, M, T, 2]
        ax, ay = anchors[..., 0], anchors[..., 1]            # [1, M, T]

        cos_t = cos_t.view(-1, 1, 1)  # [B, 1, 1]
        sin_t = sin_t.view(-1, 1, 1)

        rx = ax * cos_t - ay * sin_t  # [B, M, T]
        ry = ax * sin_t + ay * cos_t  # [B, M, T]
        return torch.stack([rx, ry], dim=-1)  # [B, M, T, 2]

    def forward(self, agent_feats, agent_valid, road_xy=None, road_type=None, road_valid=None, heading=None):
        # agent_feats [B, A, T, F], agent_valid [B, A, T], road_xy [B, R, ROAD_FEAT_DIM], heading [B, 2] or None
        B, A, T, _ = agent_feats.shape
        device = agent_feats.device

        x = self.input_proj(agent_feats)  # [B, A, T, D]
        agent_ids = torch.arange(A, device=device)
        x = x + self.agent_embed(agent_ids).view(1, A, 1, -1)

        # Positional encoding along the time axis
        x = x.permute(2, 0, 1, 3).reshape(T, B * A, -1)
        x = self.pos_encoding(x)
        x = x.reshape(T, B, A, -1).permute(1, 2, 0, 3).reshape(B, A * T, -1)
        x = x + self.modality_embed(torch.zeros(1, dtype=torch.long, device=device)).view(1, 1, -1)

        agent_padding_mask = agent_valid.reshape(B, A * T) == 0  # True = ignore this token

        if road_xy is not None and road_xy.shape[1] > 0:
            r = self.road_proj(road_xy) + self.road_type_embed(road_type)  # [B, R, D]
            r = r + self.modality_embed(torch.ones(1, dtype=torch.long, device=device)).view(1, 1, -1)
            road_padding_mask = road_valid == 0  # [B, R]

            tokens = torch.cat([x, r], dim=1)                       # [B, A*T + R, D]
            padding_mask = torch.cat([agent_padding_mask, road_padding_mask], dim=1)
        else:
            tokens = x
            padding_mask = agent_padding_mask

        tokens = tokens.permute(1, 0, 2)  # [seq_len, B, D] — encoder expects seq-first here
        memory = self.encoder(tokens, src_key_padding_mask=padding_mask)  # [seq_len, B, D]

        # Mode decoding: each query conditioned on its anchor endpoint
        anchor_ends = self.anchor_paths[:, -1, :]                                   # [M, 2]
        mode_q = self.mode_queries + self.anchor_proj(anchor_ends / POS_SCALE_M)    # [M, D]
        if self.num_groups > 0:
            mode_q = mode_q + self.group_embed(self.anchor_group)                    # shared per-group term
        queries = mode_q.unsqueeze(1).repeat(1, B, 1)  # [M, B, D]
        mode_out = self.mode_decoder(queries, memory, memory_key_padding_mask=padding_mask)  # [M, B, D]

        # Broadcast each mode across the horizon, add future positional encoding, decode per timestep
        future_pe = self.future_pos_encoding.pe[:self.future_steps]          # [T, D]
        decoder_in = mode_out.unsqueeze(2) + future_pe.view(1, 1, self.future_steps, -1)  # [M, B, T, D]
        hidden = self.decoder_mlp[:-1](decoder_in)  # [M, B, T, DECODER_FF_DIM] (bf16 under autocast)

        # Output layer, confidence head and anchor addition in fp32 for coordinate precision
        with torch.autocast(device_type=device.type, enabled=False):
            raw = self.decoder_mlp[-1](hidden.float())  # [M, B, T, 3]
            raw = raw.permute(1, 0, 2, 3)               # [B, M, T, 3]

            if self.cumulative_offsets:
                offset = torch.cumsum(raw[..., :2], dim=2)   # per-step deltas, integrated over time
            else:
                offset = raw[..., :2]                        # direct per-step offsets
            log_std = raw[..., 2].clamp(LOG_STD_MIN, LOG_STD_MAX)    # [B, M, T]

            if heading is not None:
                anchors = self._rotated_anchors(heading.float())     # [B, M, T, 2]
            else:
                anchors = self.anchor_paths.unsqueeze(0)              # [1, M, T, 2]
            trajs = anchors + offset  # anchor-relative regression

            confs = self.conf_head(mode_out.float()).squeeze(-1).permute(1, 0)  # [B, M]

        return trajs, confs, log_std

In [ ]:
# Converts one sample into padded tensors in the target's agent frame (origin = current position, +x = heading).

VELOCITY_SCALE_MPS = 20.0


def to_agent_frame(xy, origin, yaw):
    """World [..., 2] points -> agent frame (translate to origin, rotate by -yaw)."""
    d = np.asarray(xy, dtype=np.float64) - np.asarray(origin, dtype=np.float64)
    c, s = math.cos(yaw), math.sin(yaw)
    return np.stack([d[..., 0] * c + d[..., 1] * s, -d[..., 0] * s + d[..., 1] * c], axis=-1).astype(np.float32)


def rotate_to_agent_frame(v, yaw):
    """World [..., 2] direction/velocity vectors -> agent frame (rotation only)."""
    v = np.asarray(v, dtype=np.float64)
    c, s = math.cos(yaw), math.sin(yaw)
    return np.stack([v[..., 0] * c + v[..., 1] * s, -v[..., 0] * s + v[..., 1] * c], axis=-1).astype(np.float32)


def build_model_inputs(scene, max_agents=MAX_AGENTS, hist_steps=HIST_STEPS, max_road_points=MAX_ROAD_POINTS,
                       max_lights=MAX_TRAFFIC_LIGHTS):
    origin, yaw0 = scene["origin"], scene["yaw0"]
    num_agents = scene["x"].shape[0]

    feats = np.zeros((max_agents, hist_steps, AGENT_FEAT_DIM), dtype=np.float32)
    valid = np.zeros((max_agents, hist_steps), dtype=np.float32)

    for i in range(min(num_agents, max_agents)):
        m = scene["valid"][i] > 0
        pos = to_agent_frame(np.stack([scene["x"][i], scene["y"][i]], axis=-1), origin, yaw0)
        vel = rotate_to_agent_frame(np.stack([scene["vx"][i], scene["vy"][i]], axis=-1), yaw0)
        dyaw = scene["bbox_yaw"][i] - yaw0
        feats[i, :, 0:2] = pos / POS_SCALE_M
        feats[i, :, 2:4] = vel / VELOCITY_SCALE_MPS
        feats[i, :, 4] = np.cos(dyaw)
        feats[i, :, 5] = np.sin(dyaw)
        slot = AGENT_TYPE_SLOT.get(int(scene["type"][i]))
        if slot is not None:
            feats[i, :, 6 + slot] = 1.0
        feats[i, ~m] = 0.0          # zero out padding steps (-1 sentinels), they're masked anyway
        valid[i] = m

    future_valid = scene["future_valid"][0] > 0
    future_xy = to_agent_frame(np.stack([scene["future_x"][0], scene["future_y"][0]], axis=-1), origin, yaw0)
    future_xy[~future_valid] = 0.0

    road_xy_raw = scene.get("road_xy", np.zeros((0, 2), dtype=np.float32))
    road_dir_raw = scene.get("road_dir", np.zeros((0, 2), dtype=np.float32))
    road_type_raw = scene.get("road_type", np.zeros((0,), dtype=np.int64))
    road_id_raw = scene.get("road_id", np.zeros((0,), dtype=np.int64))
    n_road = min(len(road_xy_raw), max_road_points)

    n_tokens = max_road_points + max_lights
    road_feats = np.zeros((n_tokens, ROAD_FEAT_DIM), dtype=np.float32)
    road_type = np.zeros((n_tokens,), dtype=np.int64)
    road_valid = np.zeros((n_tokens,), dtype=np.float32)
    road_id = np.full((n_tokens,), -1, dtype=np.int64)

    if n_road > 0:
        road_feats[:n_road, 0:2] = to_agent_frame(road_xy_raw[:n_road], origin, yaw0) / POS_SCALE_M
        road_feats[:n_road, 2:4] = rotate_to_agent_frame(road_dir_raw[:n_road], yaw0)
        road_type[:n_road] = road_type_raw[:n_road]
        road_valid[:n_road] = 1.0
        road_id[:n_road] = road_id_raw[:n_road]

    # traffic lights occupy the slot range [max_road_points, max_road_points + max_lights)
    tl_xy_raw = scene.get("tl_xy", np.zeros((0, 2), dtype=np.float32))
    n_tl = min(len(tl_xy_raw), max_lights)
    if n_tl > 0:
        sl = slice(max_road_points, max_road_points + n_tl)
        road_feats[sl, 0:2] = to_agent_frame(tl_xy_raw[:n_tl], origin, yaw0) / POS_SCALE_M
        road_feats[sl, 4] = scene["tl_time"][:n_tl]
        road_type[sl] = scene["tl_type"][:n_tl]
        road_valid[sl] = 1.0

    return {
        "agent_feats": torch.from_numpy(feats),              # [max_agents, HIST_STEPS, AGENT_FEAT_DIM]
        "agent_valid": torch.from_numpy(valid),              # [max_agents, HIST_STEPS]
        "future_xy": torch.from_numpy(future_xy),            # [FUTURE_STEPS, 2], meters, agent frame
        "future_valid": torch.from_numpy(future_valid.astype(np.float32)),
        "road_xy": torch.from_numpy(road_feats),             # [MAX_ROAD_TOKENS, 5], map points + traffic lights
        "road_type": torch.from_numpy(road_type),            # [MAX_ROAD_TOKENS]
        "road_valid": torch.from_numpy(road_valid),          # [MAX_ROAD_TOKENS]
        "road_id": torch.from_numpy(road_id),                # [MAX_ROAD_TOKENS], viz only
        "heading": torch.tensor([1.0, 0.0]),                 # agent frame: forward is +x, anchor rotation = identity
        "origin": origin,                                    # world, for viz only
        "yaw0": yaw0,                                        # world, for viz only
    }

In [ ]:
# Loss: Gaussian NLL on the winning mode + confidence cross-entropy, with selectable assignment and confidence targets.

ASSIGNMENT_MODE = "grouped"      # "grouped", "trajectory", "endpoint", "anchor", or "soft"
SOFT_ASSIGN_TEMP_M = 0.5         # used only when ASSIGNMENT_MODE == "soft"
GROUPED_HYBRID = True            # grouped only: closest in-group slot regresses; soft in-group confidence
CONF_TARGET = "relative"         # "relative", "hard" or "fixed" (hybrid uses the in-group relative target)
REL_TAU_SCALE = 0.15
REL_TAU_BIAS = 0.2               # also the minimum tau (m)
REL_TAU_MAX = 2.0
CONF_TARGET_TEMP_M = 2.0         # only for CONF_TARGET == "fixed"
LABEL_SMOOTHING = 0.0            # only for CONF_TARGET == "hard"
SOFT_CONF_TARGET = CONF_TARGET != "hard"   # recorded in the run config
NLL_WEIGHT = 0.2                 # keeps the metre-scale NLL from swamping the confidence loss
CONF_WEIGHT = 1.0
NLL_WARMUP_EPOCHS = 10           # unused; the warmup is NLL_WARMUP_STEPS

# Per-batch diagnostics read by the training loop (detached tensors)
LOSS_STATS = {}

# Grouped anchor bank on DEVICE, set by the training cell
GROUP_ANCHORS = globals().get("GROUP_ANCHORS")
GROUP_OF_ANCHOR = globals().get("GROUP_OF_ANCHOR")


def gt_groups_torch(target_xy, target_valid):
    """GPU twin of maneuver_features + assign_groups (maneuver-group cell) -> group [B], endpoint [B, 2], last [B].
    Same forward-fill, heading from the last ~1 s, path length from the origin, same thresholds."""
    B, T, _ = target_xy.shape
    dev = target_xy.device
    valid = target_valid > 0
    t = torch.arange(T, device=dev).unsqueeze(0).expand(B, T)
    lv = torch.where(valid, t, torch.full_like(t, -1)).cummax(dim=1).values                # [B, T]
    first = valid.float().argmax(dim=1, keepdim=True)
    src = torch.where(lv < 0, first.expand(B, T), lv)
    traj = torch.gather(target_xy, 1, src.unsqueeze(-1).expand(B, T, 2))                   # forward-filled
    last = lv[:, -1].clamp(min=0)
    rows = torch.arange(B, device=dev)
    end = traj[rows, last]
    L = torch.diff(torch.cat([torch.zeros(B, 1, 2, device=dev), traj], dim=1), dim=1).norm(dim=-1).sum(1)
    tail = end - traj[rows, (last - 10).clamp(min=0)]
    tail = torch.where(tail.norm(dim=-1, keepdim=True) > 0.5, tail, end)
    deg = torch.rad2deg(torch.atan2(tail[:, 1], tail[:, 0]))
    max_lat = traj[..., 1].abs().amax(dim=1)
    g = torch.ones(B, dtype=torch.long, device=dev)                                         # straight
    g[deg > TURN_DEG] = 2
    g[deg < -TURN_DEG] = 3
    g[deg.abs() > UTURN_DEG] = 4
    g[(deg.abs() > UTURN_DEG) & (max_lat < UTURN_MIN_LAT_M)] = 0                             # backing up
    g[L < STATIONARY_LEN_M] = 0
    return g, end, last


def grouped_winner(target_xy, target_valid, anchors=None, anchor_group=None):
    """Assigned slot [B]: maneuver group from the GT, then the nearest anchor (at the GT's last valid
    step) among that group's slots. Twin of assign_slots (grouped-anchor cell)."""
    anchors = GROUP_ANCHORS if anchors is None else anchors
    anchor_group = GROUP_OF_ANCHOR if anchor_group is None else anchor_group
    if anchors is None or anchor_group is None:
        raise RuntimeError("GROUP_ANCHORS / GROUP_OF_ANCHOR not set: run the grouped-anchor cell, then the training cell.")
    g, end, last = gt_groups_torch(target_xy.float(), target_valid)
    anc_at = anchors[:, last].permute(1, 0, 2)                                              # [B, M, 2]
    d = (anc_at - end.unsqueeze(1)).norm(dim=-1)                                            # [B, M]
    d = d.masked_fill(anchor_group.unsqueeze(0) != g.unsqueeze(1), float("inf"))
    return d.argmin(dim=1)

# Set by the anchor-fit cell; kept if already set
ANCHOR_KF = globals().get("ANCHOR_KF")


def anchor_labels(target_xy, target_valid, kf=None):
    """Ground truth -> anchor index [B], matching kmeans.predict at fit time:
    forward-fill invalid steps, take (2s, 5s, last-valid) keyframes,
    min-max normalize, nearest cluster center."""
    kf = ANCHOR_KF if kf is None else kf
    if not kf:
        raise RuntimeError("ANCHOR_KF is not set (only 'position' anchors define it): "
                           "use ASSIGNMENT_MODE = 'trajectory' or fit position anchors.")
    B, T, _ = target_xy.shape
    valid = target_valid > 0
    t = torch.arange(T, device=target_xy.device).unsqueeze(0).expand(B, T)
    last_valid_le_t = torch.where(valid, t, torch.full_like(t, -1)).cummax(dim=1).values   # [B, T]
    first_valid = valid.float().argmax(dim=1, keepdim=True)                                # [B, 1]
    src = torch.where(last_valid_le_t < 0, first_valid.expand(B, T), last_valid_le_t)
    traj = torch.gather(target_xy, 1, src.unsqueeze(-1).expand(B, T, 2))                  # forward-filled

    last = last_valid_le_t[:, -1].clamp(min=0)
    rows = torch.arange(B, device=target_xy.device)
    k2 = torch.clamp(last, max=20)
    k5 = torch.clamp(last, max=50)
    feats = torch.cat([traj[rows, k2], traj[rows, k5], traj[rows, last]], dim=-1)         # [B, 6]
    norm = (feats - kf["min"]) / kf["range"]
    return torch.cdist(norm, kf["centers"]).argmin(dim=1)                                  # [B]


def confidence_target(ade, weights=None, mode=None):
    """ade [B, M] (no grad) -> confidence target [B, M]. `weights` is the regression assignment
    (one-hot for hard modes), used only by the "hard" target."""
    mode = CONF_TARGET if mode is None else mode
    if mode == "relative":
        ade_min = ade.min(dim=1, keepdim=True).values                                       # [B, 1]
        tau = (REL_TAU_SCALE * ade_min + REL_TAU_BIAS).clamp(min=REL_TAU_BIAS, max=REL_TAU_MAX)
        return F.softmax(-(ade - ade_min) / tau, dim=1)
    if mode == "fixed":
        return F.softmax(-ade / CONF_TARGET_TEMP_M, dim=1)
    if mode == "hard":
        M = ade.shape[1]
        if weights is None:
            weights = F.one_hot(ade.argmin(dim=1), M).float()
        return (1.0 - LABEL_SMOOTHING) * weights + LABEL_SMOOTHING / M
    raise ValueError(f"Unknown CONF_TARGET: {mode}")


def gaussian_nll_loss(pred_trajs, pred_confs, pred_log_std, target_xy, target_valid, use_learned_std=True):
    """Returns (total_loss, nll_loss, conf_loss); the last two are detached tensors (unweighted)."""
    # pred_trajs [B, M, T, 2], pred_confs [B, M], pred_log_std [B, M, T]; target_xy [B, T, 2], target_valid [B, T]
    B, M, T, _ = pred_trajs.shape
    batch_idx = torch.arange(B, device=pred_trajs.device)

    sq_dist = ((pred_trajs - target_xy.unsqueeze(1)) ** 2).sum(-1)   # [B, M, T]
    vmask = target_valid.unsqueeze(1)                                  # [B, 1, T]
    denom = target_valid.sum(-1).clamp(min=1.0).unsqueeze(1)           # [B, 1]

    # Isotropic 2D Gaussian NLL per mode and timestep (constant term dropped)
    log_std = pred_log_std if use_learned_std else torch.zeros_like(pred_log_std)  # sigma=1 during warmup
    nll_per_t = 2.0 * log_std + sq_dist * torch.exp(-2.0 * log_std) / 2.0         # [B, M, T]
    nll_per_mode = (nll_per_t * vmask).sum(-1) / denom                             # [B, M]

    # ADE of every mode to the truth (no gradient), used for winners and soft targets
    with torch.no_grad():
        ade = (sq_dist.clamp(min=0.0).sqrt() * vmask).sum(-1) / denom             # [B, M]

    winner = None
    assigned = None          # grouped only: the anchor-assigned slot (from the GT)
    same_group = None        # grouped only: [B, M] True for slots in the GT's group
    if ASSIGNMENT_MODE == "soft":
        # soft responsibilities are fixed targets (no gradient through the weights)
        weights = F.softmax(-ade / SOFT_ASSIGN_TEMP_M, dim=1)                      # [B, M]
    else:
        if ASSIGNMENT_MODE == "grouped":
            with torch.no_grad():
                assigned = grouped_winner(target_xy, target_valid)                 # [B], from GT only
                same_group = GROUP_OF_ANCHOR.unsqueeze(0) == GROUP_OF_ANCHOR[assigned].unsqueeze(1)  # [B, M]
                if GROUPED_HYBRID:
                    # closest predicted slot within the GT's maneuver group
                    winner = ade.masked_fill(~same_group, float("inf")).argmin(dim=1)              # [B]
                else:
                    winner = assigned
        elif ASSIGNMENT_MODE == "trajectory":
            winner = ade.argmin(dim=1)                                             # [B]
        elif ASSIGNMENT_MODE == "anchor":
            winner = anchor_labels(target_xy, target_valid)                        # [B]
        elif ASSIGNMENT_MODE == "endpoint":
            # last valid step (validity is not always a contiguous prefix)
            time_idx = torch.arange(T, device=target_valid.device)
            last_idx = (target_valid * time_idx.unsqueeze(0)).max(dim=1).values.long()  # [B]
            winner = sq_dist[batch_idx, :, last_idx].argmin(dim=1)                      # [B]
        else:
            raise ValueError(f"Unknown ASSIGNMENT_MODE: {ASSIGNMENT_MODE}")
        weights = F.one_hot(winner, M).float()

    nll_loss = (weights * nll_per_mode).sum(dim=1).mean()

    with torch.no_grad():
        if ASSIGNMENT_MODE == "grouped" and GROUPED_HYBRID:
            # hybrid target: zero outside the GT group, relative-tau softmax over ADE inside it
            ade_g = ade.masked_fill(~same_group, float("inf"))
            ade_min = ade_g.min(dim=1, keepdim=True).values
            tau = (REL_TAU_SCALE * ade_min + REL_TAU_BIAS).clamp(min=REL_TAU_BIAS, max=REL_TAU_MAX)
            conf_target = F.softmax(-(ade_g - ade_min) / tau, dim=1)              # zero outside the group
        else:
            conf_target = confidence_target(ade, weights)                         # [B, M]
    conf_loss = -(conf_target * F.log_softmax(pred_confs, dim=-1)).sum(dim=-1).mean()

    total_loss = NLL_WEIGHT * nll_loss + CONF_WEIGHT * conf_loss

    with torch.no_grad():
        LOSS_STATS["p_top1"] = F.softmax(pred_confs, dim=-1).max(dim=1).values.mean()
        LOSS_STATS["tgt_entropy"] = -(conf_target * conf_target.clamp(min=1e-12).log()).sum(-1).mean()
        LOSS_STATS["tgt_win"] = conf_target.gather(1, ade.argmin(dim=1, keepdim=True)).mean()
        if ASSIGNMENT_MODE == "grouped":
            top1 = pred_confs.argmax(dim=1)
            LOSS_STATS["assigned_top1"] = (top1 == assigned).float().mean()
            LOSS_STATS["group_top1"] = same_group.gather(1, top1.unsqueeze(1)).float().mean()
            LOSS_STATS["reg_is_assigned"] = (winner == assigned).float().mean()
            LOSS_STATS["rank_top1"] = LOSS_STATS["group_top1"] if GROUPED_HYBRID else LOSS_STATS["assigned_top1"]

    return total_loss, nll_loss.detach(), conf_loss.detach()


def collate_scenes(batch):
    # Cast compactly stored samples to model dtypes, per batch.
    feats = torch.stack([s["agent_feats"] for s in batch]).float()
    valid = torch.stack([s["agent_valid"] for s in batch]).float()
    future_xy = torch.stack([s["future_xy"] for s in batch]).float()
    future_valid = torch.stack([s["future_valid"] for s in batch]).float()
    road_xy = torch.stack([s["road_xy"] for s in batch]).float()
    road_type = torch.stack([s["road_type"] for s in batch]).long()
    road_valid = torch.stack([s["road_valid"] for s in batch]).float()
    heading = torch.stack([s["heading"] for s in batch]).float()  # [B, 2], always [1, 0] in the agent frame
    return feats, valid, future_xy, future_valid, road_xy, road_type, road_valid, heading

In [ ]:
# Data: tensorize shards to Drive once (CPU runtime), then load compact tensors into RAM; split by shard id % 10.

import json
import subprocess
from google.colab import drive
drive.mount("/content/drive")

SMOKE_TEST = False   # True: 4-shard smoke set; False: all 600 shards

SEED = 42
if SMOKE_TEST:
    PREP_OUT_DIR = "/content/drive/MyDrive/waymo_trajectory/prepped_smoke"
    PREP_SHARD_IDS = [0, 1, 8, 9]       # 2 train, 1 val (id%10==8), 1 test (id%10==9)
else:
    PREP_OUT_DIR = "/content/drive/MyDrive/waymo_trajectory/prepped_600"
    PREP_SHARD_IDS = list(range(600))
DOWNLOAD_BATCH = 8                     # shards per parallel gsutil -m call (bounds local disk use)
DOWNLOAD_RETRIES = 3
SHARDS_PER_CHUNK = 20                  # shards accumulated per saved .pt chunk
MAX_STEP_SPEED_MPS = 45.0              # drop a sample if any consecutive valid future step exceeds this
GCS_PREFIX = "gs://waymo_open_dataset_motion_v_1_2_0/uncompressed/tf_example/training"
FORCE_PREP = False                     # set True to prep even on a GPU runtime
os.makedirs(PREP_OUT_DIR, exist_ok=True)
DONE_FLAG = os.path.join(PREP_OUT_DIR, "PREP_DONE")

# Shard-count summary for the run config
n_shards = len(PREP_SHARD_IDS)
n_val_shards = sum(1 for i in PREP_SHARD_IDS if i % 10 == 8)
n_test_shards = sum(1 for i in PREP_SHARD_IDS if i % 10 == 9)
n_train_shards = n_shards - n_val_shards - n_test_shards


def _read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def _write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f)


def _remote(i): return f"{GCS_PREFIX}/training_tfexample.tfrecord-{i:05d}-of-01000"
def _local(i):  return f"/content/training_tfexample.tfrecord-{i:05d}-of-01000"


def _shard_split(shard_id):
    r = shard_id % 10
    return "test" if r == 9 else ("val" if r == 8 else "train")


def _velocity_ok(inp):
    # Drop physically impossible tracks (implied speed between consecutive valid future steps above the threshold)
    fx = inp["future_xy"].numpy()
    fv = inp["future_valid"].numpy() > 0
    step = np.linalg.norm(np.diff(fx, axis=0), axis=1)          # [T-1] metres between adjacent steps
    both = fv[:-1] & fv[1:]
    return not (both.any() and (step[both] / 0.1 > MAX_STEP_SPEED_MPS).any())


def _pack(s):   # compact storage: fp16 features, fp32 targets, uint8 masks, int16 road types
    return {"agent_feats": s["agent_feats"].half(),
            "agent_valid": (s["agent_valid"] > 0).to(torch.uint8),
            "future_xy": s["future_xy"].float(),
            "future_valid": (s["future_valid"] > 0).to(torch.uint8),
            "road_xy": s["road_xy"].half(),
            "road_type": s["road_type"].to(torch.int16),
            "road_valid": (s["road_valid"] > 0).to(torch.uint8)}


def run_prep():
    n_chunks = (len(PREP_SHARD_IDS) + SHARDS_PER_CHUNK - 1) // SHARDS_PER_CHUNK
    incomplete = []
    shape_checked = False
    for cidx in range(n_chunks):
        shard_batch = PREP_SHARD_IDS[cidx * SHARDS_PER_CHUNK:(cidx + 1) * SHARDS_PER_CHUNK]
        marker = os.path.join(PREP_OUT_DIR, f"chunk_{cidx:03d}_done")
        if _read_json(marker) == shard_batch:   # done only if built from exactly these shards
            print(f"chunk {cidx}: already saved, skipping"); continue
        buf = {"train": [], "val": [], "test": []}
        n_dropped, missing = 0, []
        t0 = time.time()
        for b0 in range(0, len(shard_batch), DOWNLOAD_BATCH):
            ids = shard_batch[b0:b0 + DOWNLOAD_BATCH]
            for _attempt in range(DOWNLOAD_RETRIES):
                need = [i for i in ids if not os.path.exists(_local(i))]
                if not need:
                    break
                subprocess.run(["gsutil", "-m", "cp", *[_remote(i) for i in need], "/content/"], check=False)
            for i in ids:
                if not os.path.exists(_local(i)):
                    missing.append(i); continue
                split = _shard_split(i)
                for parsed in load_dataset(_local(i)):
                    for scene in extract_scene_agents(parsed, max_targets=10_000):
                        inp = build_model_inputs(scene)
                        if not shape_checked:
                            # Guard against stale cells: stored shapes must match the current constants
                            want = (MAX_AGENTS, HIST_STEPS, AGENT_FEAT_DIM)
                            if tuple(inp["agent_feats"].shape) != want:
                                raise RuntimeError(f"agent_feats is {tuple(inp['agent_feats'].shape)}, expected {want}. "
                                                   "Re-run the constants, extraction and build_model_inputs cells, "
                                                   "then this one.")
                            shape_checked = True
                        if inp["future_valid"].sum() == 0:
                            continue
                        if not _velocity_ok(inp):
                            n_dropped += 1
                            continue
                        buf[split].append(_pack(inp))
                os.remove(_local(i))
        counts = {}
        for split in ("train", "val", "test"):
            fpath = os.path.join(PREP_OUT_DIR, f"chunk_{cidx:03d}_{split}.pt")
            if buf[split]:
                stacked = {k: torch.stack([smp[k] for smp in buf[split]]) for k in buf[split][0]}
                torch.save(stacked, fpath)
                counts[split] = len(buf[split])
            elif os.path.exists(fpath):
                os.remove(fpath)   # stale file from an earlier partial run
        if missing:
            incomplete.append(cidx)
            print(f"chunk {cidx}: {counts} -- NOT marked done, failed to download shards {missing}")
        else:
            _write_json(marker, shard_batch)
            print(f"chunk {cidx}: {counts}, dropped {n_dropped} by speed check, "
                  f"in {(time.time() - t0) / 60:.1f} min")
    if incomplete:
        print(f"INCOMPLETE: chunks {incomplete} are missing shards. Check gsutil auth "
              f"(re-run the auth cell if needed), then re-run this cell -- finished chunks are skipped.")
    else:
        _write_json(DONE_FLAG, PREP_SHARD_IDS)
        print("PREP COMPLETE. Re-run this cell to load (High-RAM A100 for the full set).")


def load_split(split):
    files = sorted(f for f in os.listdir(PREP_OUT_DIR) if f.endswith(f"_{split}.pt"))
    if not files:
        print(f"WARNING: no '{split}' files in {PREP_OUT_DIR} -- this split is empty.")
        return []
    acc = {}
    t0 = time.time()
    for j, f in enumerate(files):
        st = torch.load(os.path.join(PREP_OUT_DIR, f))
        for k, v in st.items():
            acc.setdefault(k, []).append(v)
        if (j + 1) % 5 == 0 or j + 1 == len(files):
            print(f"  {split}: read {j + 1}/{len(files)} files ({(time.time() - t0) / 60:.1f} min)", flush=True)
    acc = {k: torch.cat(v) for k, v in acc.items()}
    h = torch.tensor([1.0, 0.0])
    n = len(acc["agent_feats"])
    # Views into the compact tensors; collate_scenes casts each batch
    return [{k: acc[k][i] for k in acc} | {"heading": h} for i in range(n)]


if os.path.exists(DONE_FLAG) and not FORCE_PREP:
    if _read_json(DONE_FLAG) != PREP_SHARD_IDS:
        raise RuntimeError(f"{PREP_OUT_DIR} was not built from the current PREP_SHARD_IDS "
                           f"({len(PREP_SHARD_IDS)} shards). Delete that folder in Drive and re-run to prep.")
    t0 = time.time()
    train_scenes = load_split("train")
    val_scenes = load_split("val")
    test_scenes = load_split("test")
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    random.shuffle(train_scenes)   # seeded -> reproducible anchors and mode indices across runs
    print(f"loaded train {len(train_scenes)}, val {len(val_scenes)}, test {len(test_scenes)} "
          f"in {(time.time() - t0) / 60:.1f} min")

    def _ttype(s):
        oh = s["agent_feats"][0, -1, 6:9].float().numpy()
        return int(np.argmax(oh)) if oh.sum() > 0 else -1
    for _name, _split in (("train", train_scenes), ("val", val_scenes)):
        if _split:
            _ty = np.array([_ttype(s) for s in _split[:5000]])
            print(f"{_name} target types: vehicle {np.mean(_ty == 0):.1%}, pedestrian {np.mean(_ty == 1):.1%}, "
                  f"cyclist {np.mean(_ty == 2):.1%}")
    if train_scenes:
        # Share of samples whose last valid point is beyond 150 m
        _end = []
        for s in train_scenes[:20000]:
            _v = np.where(s["future_valid"].numpy() > 0)[0]
            _end.append(float(np.linalg.norm(s["future_xy"][_v[-1]].numpy())))
        _end = np.array(_end)
        print(f"train endpoints beyond 150 m: {np.mean(_end > 150):.2%}, "
              f"max {_end.max():.0f} m")
        _s = train_scenes[0]
        print("sample shapes/dtypes:", {k: (tuple(v.shape), str(v.dtype).replace("torch.", "")) for k, v in _s.items()})
        _b = collate_scenes(train_scenes[:4])
        print("collated batch dtypes:", [str(t.dtype).replace("torch.", "") for t in _b])
elif torch.cuda.is_available() and not FORCE_PREP:
    print("No prepped data found and this is a GPU runtime. Run prep on a CPU runtime first "
          "(Runtime > Change runtime type > CPU, then run this cell), then return to the A100 to load. "
          "Set FORCE_PREP = True to override.")
else:
    run_prep()

In [ ]:
# Base shape anchors and shared helpers (_futures, _fit_set); the grouped anchors below are the ones used for training.

ANCHOR_FIT_N = 200_000
ANCHOR_MAX_DIST_M = 400.0       # ignore endpoints farther than this (glitches) when fitting
ANCHOR_SPACE = "shape"
STATIONARY_LEN_M = 2.0
N_STRAIGHT_ANCHORS = 2
STRAIGHT_MAX_DEG = 10.0         # straight: final heading within this of +x ...
STRAIGHT_MAX_LAT_M = 1.5        # ... and within this lateral distance


def _futures(scenes):
    """Stack futures -> forward-filled paths [N, T, 2], valid [N, T] bool, last valid index [N]."""
    xy = torch.stack([s["future_xy"] for s in scenes]).float().numpy()
    fv = torch.stack([s["future_valid"] for s in scenes]).numpy() > 0
    N, T, _ = xy.shape
    t = np.arange(T)
    idx = np.maximum.accumulate(np.where(fv, t, -1), axis=1)
    first = fv.argmax(axis=1)
    idx = np.where(idx < 0, first[:, None], idx)
    paths = np.take_along_axis(xy, idx[..., None], axis=1)
    last = np.where(fv, t, -1).max(axis=1)
    return paths, fv, last


def _medoid(features, members, center):
    return members[np.argmin(np.linalg.norm(features[members] - center, axis=1))]


def fit_anchor_paths(scenes, num_modes=NUM_MODES):
    paths, fv, last = _futures(scenes)
    N, T, _ = paths.shape
    rows = np.arange(N)
    end = paths[rows, last]
    k2, k5 = paths[rows, np.minimum(20, last)], paths[rows, np.minimum(50, last)]
    full = last == T - 1
    inlier = full & (np.linalg.norm(end, axis=1) <= ANCHOR_MAX_DIST_M)
    print(f"futures valid for the full 8 s: {full.mean():.1%} of {N} (only these are used to fit)")

    if ANCHOR_SPACE == "position":
        shapes = np.concatenate([k2, k5, end], axis=1)
        lo, hi = shapes[inlier].min(0), shapes[inlier].max(0)
        rng = np.clip(hi - lo, 1e-3, None)
        feats = (shapes - lo) / rng
        km = KMeans(n_clusters=num_modes, n_init="auto", random_state=0).fit(feats[inlier])
        labels = km.predict(feats)
        anchors = np.zeros((num_modes, T, 2), np.float32)
        for m in range(num_modes):
            mem = np.where((labels == m) & inlier)[0]
            if len(mem):
                anchors[m] = paths[_medoid(feats, mem, km.cluster_centers_[m])]
        kf = {"centers": torch.tensor(km.cluster_centers_, dtype=torch.float32),
              "min": torch.tensor(lo, dtype=torch.float32), "range": torch.tensor(rng, dtype=torch.float32)}
        print(f"position anchors: k-means on {inlier.sum()} paths")
        return torch.from_numpy(anchors), kf

    # --- shape (intent) anchors ---
    steps = np.linalg.norm(np.diff(np.concatenate([np.zeros((N, 1, 2), np.float32), paths], axis=1), axis=1), axis=-1)
    L = steps.sum(axis=1)                                                     # path length over the 8 s
    tail = end - paths[rows, np.maximum(last - 10, 0)]                        # last ~1 s of motion
    tail = np.where((np.linalg.norm(tail, axis=1) > 0.5)[:, None], tail, end)
    theta = np.arctan2(tail[:, 1], tail[:, 0])                                # final heading
    max_lat = np.abs(paths[..., 1]).max(axis=1)

    stationary = inlier & (L < STATIONARY_LEN_M)
    moving = inlier & ~stationary
    straight = moving & (np.abs(np.degrees(theta)) < STRAIGHT_MAX_DEG) & (max_lat < STRAIGHT_MAX_LAT_M)
    other = moving & ~straight

    anchors = [np.zeros((T, 2), np.float32)]                                  # anchor 0: stationary
    s_idx = np.where(straight)[0]
    edges = np.quantile(L[s_idx], np.linspace(0, 1, N_STRAIGHT_ANCHORS + 1))
    for b in range(N_STRAIGHT_ANCHORS):
        lo_e, hi_e = edges[b], edges[b + 1]
        mem = s_idx[(L[s_idx] >= lo_e) & ((L[s_idx] <= hi_e) if b == N_STRAIGHT_ANCHORS - 1 else (L[s_idx] < hi_e))]
        med = np.median(L[mem])
        anchors.append(paths[mem[np.argmin(np.abs(L[mem] - med) + max_lat[mem])]])   # near-median length, straightest

    k_shape = num_modes - 1 - N_STRAIGHT_ANCHORS
    o_idx = np.where(other)[0]
    Lo = np.maximum(L[o_idx], 1e-3)[:, None]
    feats = np.concatenate([k2[o_idx] / Lo, k5[o_idx] / Lo, end[o_idx] / Lo,
                            np.cos(theta[o_idx])[:, None], np.sin(theta[o_idx])[:, None]], axis=1)
    km = KMeans(n_clusters=k_shape, n_init="auto", random_state=0).fit(feats)
    shape_anchors = []
    for m in range(k_shape):
        mem = np.where(km.labels_ == m)[0]
        oi = o_idx[_medoid(feats, mem, km.cluster_centers_[m])]
        shape_anchors.append((theta[oi], paths[oi]))
    shape_anchors.sort(key=lambda a: a[0])                                    # right turns ... left turns
    anchors += [p for _, p in shape_anchors]

    n_in = inlier.sum()
    print(f"shape anchors fit on {n_in} full-length paths: stationary {stationary.sum() / n_in:.1%}, "
          f"straight {straight.sum() / n_in:.1%} ({N_STRAIGHT_ANCHORS} length bins, edges "
          f"{', '.join(f'{e:.0f}' for e in edges)} m), other {other.sum() / n_in:.1%} ({k_shape} shape clusters)")
    return torch.from_numpy(np.stack(anchors).astype(np.float32)), {}


_fit_set = train_scenes[:ANCHOR_FIT_N]   # train_scenes was shuffled with SEED in the data cell
anchor_paths, _kf_cpu = fit_anchor_paths(_fit_set)
ANCHOR_KF = {k: v.to(DEVICE) for k, v in _kf_cpu.items()}

for m in range(NUM_MODES):
    p = anchor_paths[m].numpy()
    seg = p[-1] - p[-11]
    L_m = np.linalg.norm(np.diff(np.concatenate([np.zeros((1, 2)), p]), axis=0), axis=1).sum()
    print(f"anchor {m:2d}: endpoint ({p[-1, 0]:6.1f}, {p[-1, 1]:6.1f}) m, path length {L_m:5.1f} m, "
          f"final heading {math.degrees(math.atan2(seg[1], seg[0])) if np.linalg.norm(seg) > 0.1 else 0.0:7.1f} deg")

# Nearest-anchor share per anchor (dead anchors show ~0%)
_anc = anchor_paths.to(DEVICE)
for name, split in (("train", train_scenes[:50_000]), ("val", val_scenes[:50_000])):
    near = []
    for i in range(0, len(split), 4096):
        chunk = split[i:i + 4096]
        fx = torch.stack([s["future_xy"] for s in chunk]).float().to(DEVICE)
        fvm = torch.stack([s["future_valid"] for s in chunk]).float().to(DEVICE)
        d = (fx[:, None] - _anc[None]).norm(dim=-1)                                  # [B, M, T]
        ade = (d * fvm[:, None]).sum(-1) / fvm.sum(-1, keepdim=True).clamp(min=1)
        near.append(ade.argmin(1).cpu())
    share = np.bincount(torch.cat(near).numpy(), minlength=NUM_MODES) / len(split)
    print(f"{name} nearest-anchor share: {np.round(share, 3).tolist()}  (anchors under 0.2%: {(share < 0.002).sum()})")

fig, ax = plt.subplots(figsize=(6, 6))
for m in range(NUM_MODES):
    ax.plot(anchor_paths[m, :, 0], anchor_paths[m, :, 1], "-", lw=1.5)
    ax.text(anchor_paths[m, -1, 0], anchor_paths[m, -1, 1], str(m), fontsize=7)
ax.set_title(f"Fitted anchors ({ANCHOR_SPACE}; agent frame, forward = +x)")
ax.set_xlabel("x (m, forward)")
ax.set_ylabel("y (m, left)")
ax.set_aspect("equal")
plt.show()

In [ ]:
# Plot the anchors over a few validation scenes (wide map, agent frame).

VIZ_ROAD_RADIUS_M = 150.0
VIZ_MAX_ROAD_POINTS = 2000
N_ANCHOR_VIZ_SAMPLES = 3

ds_anchor_viz = load_dataset(val_paths).shuffle(500, seed=random.randint(0, 1_000_000))
n_shown = 0
for parsed_scene in ds_anchor_viz.take(50):
    samples = extract_scene_agents(parsed_scene, max_targets=1)
    if not samples:
        continue
    scene = samples[0]
    scene_inputs = build_model_inputs(scene)

    wide_xy_raw, wide_type, wide_id, _ = extract_road_graph(
        parsed_scene, scene["origin"], radius=VIZ_ROAD_RADIUS_M, max_points=VIZ_MAX_ROAD_POINTS, square=True
    )
    wide_xy = to_agent_frame(wide_xy_raw, scene["origin"], scene["yaw0"])

    fig, ax = plt.subplots(figsize=(7, 7))
    draw_road_graph(ax, wide_xy, wide_type, wide_id)

    past_xy = scene_inputs["agent_feats"].numpy()[0, :, :2] * POS_SCALE_M
    past_valid = scene_inputs["agent_valid"].numpy()[0] > 0
    ax.plot(past_xy[past_valid, 0], past_xy[past_valid, 1], "o-", color="crimson",
            linewidth=2, zorder=3, label="target (past)")
    for m in range(NUM_MODES):
        ax.plot(anchor_paths[m, :, 0], anchor_paths[m, :, 1], "-o", markersize=2, label=f"anchor {m}")
    ax.annotate("", xy=(15, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="black", lw=2))
    ax.annotate("yaw", xy=(16, 0), fontsize=8)

    ax.set_xlim(-60, 140)
    ax.set_ylim(-100, 100)
    ax.set_title(f"VAL scene {n_shown}: anchors in agent frame")
    ax.set_xlabel("x (m, forward)")
    ax.set_ylabel("y (m, left)")
    ax.legend(fontsize=6, loc="best")
    ax.set_aspect("equal")
    plt.show()

    n_shown += 1
    if n_shown >= N_ANCHOR_VIZ_SAMPLES:
        break

In [ ]:
# Print each anchor's max |x|, max |y| and endpoint distance.
for m in range(NUM_MODES):
    path = anchor_paths[m].numpy()
    max_x = np.abs(path[:, 0]).max()
    max_y = np.abs(path[:, 1]).max()
    endpoint_dist = np.linalg.norm(path[-1])
    print(f"Anchor {m}: Max |X| = {max_x:.1f}m, Max |Y| = {max_y:.1f}m, Endpoint Distance = {endpoint_dist:.1f}m")

In [ ]:
# Maneuver groups from ground-truth geometry (stationary/reverse, straight, left, right, U-turn), with diagnostics.

GROUP_NAMES = ["stationary/reverse", "straight", "left turn", "right turn", "U-turn"]
GROUP_COLORS = ["#7f7f7f", "#1f77b4", "#2ca02c", "#d62728", "#9467bd"]
TURN_DEG = 35.0
UTURN_DEG = 135.0
LANE_CHANGE_R_M = 1.75          # diagnostic only (dashed lines on the r histogram)
UTURN_MIN_LAT_M = 4.0           # a real U-turn swings sideways; reversing in place does not


def maneuver_features(paths, last):
    """paths [N, T, 2] forward-filled, last [N] last valid index -> theta (rad), L (m), r (m), max |y| (m)."""
    N = paths.shape[0]
    rows = np.arange(N)
    end = paths[rows, last]
    steps = np.linalg.norm(np.diff(np.concatenate([np.zeros((N, 1, 2), np.float32), paths], axis=1), axis=1), axis=-1)
    L = steps.sum(axis=1)
    tail = end - paths[rows, np.maximum(last - 10, 0)]
    tail = np.where((np.linalg.norm(tail, axis=1) > 0.5)[:, None], tail, end)
    theta = np.arctan2(tail[:, 1], tail[:, 0])
    small = np.abs(theta) < 1e-3
    d_arc = np.where(small, L * theta / 2, L * (1 - np.cos(theta)) / np.where(small, 1.0, theta))
    r = end[:, 1] - d_arc
    max_lat = np.abs(paths[..., 1]).max(axis=1)
    return theta, L, r, max_lat


def assign_groups(theta, L, r, max_lat):
    deg = np.degrees(theta)
    g = np.full(len(theta), 1, dtype=np.int64)                                 # straight
    g[deg > TURN_DEG] = 2
    g[deg < -TURN_DEG] = 3
    g[np.abs(deg) > UTURN_DEG] = 4
    g[(np.abs(deg) > UTURN_DEG) & (max_lat < UTURN_MIN_LAT_M)] = 0             # backing up, not a U-turn
    g[L < STATIONARY_LEN_M] = 0
    return g


def _type_of(s):
    oh = s["agent_feats"][0, -1, 6:9].float().numpy()
    return int(np.argmax(oh)) if oh.sum() > 0 else -1


# 1. Group shares and the heading / lateral-residual distributions
_gp, _gfv, _glast = _futures(_fit_set)
_gth, _gL, _gr, _gml = maneuver_features(_gp, _glast)
gt_groups = assign_groups(_gth, _gL, _gr, _gml)
_full = _glast == _gp.shape[1] - 1
_types = np.array([_type_of(s) for s in _fit_set])

print(f"GT futures: {len(gt_groups)} (first ANCHOR_FIT_N of the seeded train shuffle); "
      f"truncated (< 8 s) {(~_full).mean():.1%}")
print(f"{'group':<20}{'all':>8}{'full 8 s':>10}{'vehicle':>10}{'ped':>8}{'cyclist':>9}")
for k, name in enumerate(GROUP_NAMES):
    row = [(gt_groups == k).mean(), (gt_groups[_full] == k).mean()]
    row += [(gt_groups[_types == t] == k).mean() for t in range(3)]
    print(f"{name:<20}{row[0]:>8.1%}{row[1]:>10.1%}{row[2]:>10.1%}{row[3]:>8.1%}{row[4]:>9.1%}")

_mov = _gL >= STATIONARY_LEN_M
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].hist(np.degrees(_gth[_mov]), bins=180, range=(-180, 180), color="#1f77b4")
for x in (-UTURN_DEG, -TURN_DEG, TURN_DEG, UTURN_DEG):
    axs[0].axvline(x, color="k", ls="--", lw=0.8)
axs[0].set_yscale("log"); axs[0].set_xlabel("final heading theta (deg)"); axs[0].set_ylabel("count (log)")
axs[0].set_title("Final heading, moving agents")
_nt = _mov & (np.abs(np.degrees(_gth)) <= TURN_DEG)
axs[1].hist(np.clip(_gr[_nt], -10, 10), bins=200, range=(-10, 10), color="#17becf")
for x in (-LANE_CHANGE_R_M, LANE_CHANGE_R_M):
    axs[1].axvline(x, color="k", ls="--", lw=0.8)
axs[1].set_yscale("log"); axs[1].set_xlabel("lateral residual r = end_y - d_arc (m), clipped to +-10")
axs[1].set_title(f"Lateral residual, |theta| <= {TURN_DEG:.0f} deg (lane changes count as straight)")
plt.tight_layout(); plt.show()

# 2. Base anchors under the same rule
_ap = anchor_paths.numpy()
_ath, _aL, _ar, _aml = maneuver_features(_ap, np.full(NUM_MODES, _ap.shape[1] - 1))
anchor_groups = assign_groups(_ath, _aL, _ar, _aml)
print("\nBase anchors per group (vs. share of GT futures):")
for k, name in enumerate(GROUP_NAMES):
    ids = np.where(anchor_groups == k)[0].tolist()
    print(f"  {name:<20} {len(ids):>2} anchors  ({(gt_groups == k).mean():5.1%} of GT)  {ids}")

fig, axs = plt.subplots(1, 2, figsize=(14, 7))
for ax, lim in zip(axs, (None, 30)):
    for m in range(NUM_MODES):
        g = anchor_groups[m]
        ax.plot(_ap[m, :, 0], _ap[m, :, 1], "-", color=GROUP_COLORS[g], lw=1.8)
        ax.plot(_ap[m, -1, 0], _ap[m, -1, 1], "o", color=GROUP_COLORS[g], ms=4)
        if lim is None or (abs(_ap[m, -1, 0]) < lim and abs(_ap[m, -1, 1]) < lim):
            ax.text(_ap[m, -1, 0], _ap[m, -1, 1], f" {m}", fontsize=8)
    handles = [plt.Line2D([], [], color=GROUP_COLORS[k], lw=3,
                          label=f"{GROUP_NAMES[k]} ({(anchor_groups == k).sum()})")
               for k in range(len(GROUP_NAMES))]
    ax.legend(handles=handles, fontsize=8, loc="upper left")
    if lim is not None:
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel("x (m, forward)"); ax.set_ylabel("y (m, left)")
    ax.set_title("Base anchors by maneuver group" + ("" if lim is None else f" (zoom +-{lim} m)"))
plt.tight_layout(); plt.show()

In [ ]:
# Grouped anchors: per-group k-means on 2 / 5 / 8 s positions; slots 2 / 12 / 8 / 8 / 2 (stationary/reverse, straight, left, right, U-turn).

GROUP_SLOTS = [2, 12, 8, 8, 2]                       # same order as GROUP_NAMES; must sum to NUM_MODES
GROUP_KMEANS_STANDARDIZE = False                     # scale k-means features per group (False = raw metres)
assert sum(GROUP_SLOTS) == NUM_MODES and len(GROUP_SLOTS) == len(GROUP_NAMES)


def fit_grouped_anchors(paths, last, groups, theta, L, inlier):
    T = paths.shape[1]
    rows = np.arange(len(paths))
    k2, k5 = paths[rows, np.minimum(20, last)], paths[rows, np.minimum(50, last)]
    end = paths[rows, last]
    feats = np.concatenate([k2, k5, end], axis=1)
    anchors, owner = [], []
    for g, n in enumerate(GROUP_SLOTS):
        idx = np.where(inlier & (groups == g))[0]
        if g == 0:                                     # stationary/reverse: zero path + reversing medoid
            group_paths = [np.zeros((T, 2), np.float32)]
            rev = idx[L[idx] >= STATIONARY_LEN_M]
            if len(rev) and n > 1:
                c = feats[rev].mean(0)
                group_paths.append(paths[rev[np.argmin(np.linalg.norm(feats[rev] - c, axis=1))]])
            while len(group_paths) < n:
                group_paths.append(np.zeros((T, 2), np.float32))
        else:
            f = feats[idx]
            if GROUP_KMEANS_STANDARDIZE:
                f = (f - f.mean(0)) / (f.std(0) + 1e-6)
            km = KMeans(n_clusters=n, n_init="auto", random_state=0).fit(f)
            group_paths = []
            for m in range(n):
                loc = np.where(km.labels_ == m)[0]
                med = loc[np.argmin(np.linalg.norm(f[loc] - km.cluster_centers_[m], axis=1))]
                group_paths.append(paths[idx[med]])
            # order slots inside a group by final heading, then length
            def _key(p):
                seg = p[-1] - p[-11]
                return (round(math.degrees(math.atan2(seg[1], seg[0])) / 10), np.linalg.norm(p[-1]))
            group_paths.sort(key=_key)
        anchors += group_paths
        owner += [g] * n
        print(f"  {GROUP_NAMES[g]:<20} {n:>2} slots, fit on {len(idx):>6} full-length paths")
    return np.stack(anchors).astype(np.float32), np.array(owner)


def assign_slots(paths, last, groups, anchors, anchor_group, grouped=True):
    """Winner slot per sample: (group, then) nearest anchor point at the GT's last valid step."""
    rows = np.arange(len(paths))
    end = paths[rows, last]                                        # [N, 2]
    anc_at = anchors[:, last].transpose(1, 0, 2)                   # [N, M, 2]
    d = np.linalg.norm(anc_at - end[:, None], axis=-1)             # [N, M]
    if grouped:
        d = np.where(anchor_group[None] == groups[:, None], d, np.inf)
    w = d.argmin(1)
    return w, d[rows, w]


_inl = (_glast == _gp.shape[1] - 1) & (np.linalg.norm(_gp[np.arange(len(_gp)), _glast], axis=1) <= ANCHOR_MAX_DIST_M)
print("Fitting grouped anchors:")
_ga, _gown = fit_grouped_anchors(_gp, _glast, gt_groups, _gth, _gL, _inl)
grouped_anchor_paths = torch.from_numpy(_ga)
ANCHOR_GROUP = torch.from_numpy(_gown)

# Assigned-slot preview on the fit set
_w, _wd = assign_slots(_gp, _glast, gt_groups, _ga, _gown, grouped=True)
_wu, _wud = assign_slots(_gp, _glast, gt_groups, _ga, _gown, grouped=False)
share = np.bincount(_w, minlength=NUM_MODES) / len(_w)
print(f"\nGrouped winner: mean endpoint distance to assigned anchor {_wd.mean():.2f} m "
      f"(median {np.median(_wd):.2f}); unconstrained nearest anchor {_wud.mean():.2f} m "
      f"(median {np.median(_wud):.2f}); group rule overrides the nearest anchor in {(_w != _wu).mean():.1%}")
print(f"{'slot':>4}  {'group':<20}{'endpoint (m)':>18}{'length':>9}{'heading':>9}{'winner share':>14}")
for m in range(NUM_MODES):
    p = _ga[m]
    seg = p[-1] - p[-11]
    hd = math.degrees(math.atan2(seg[1], seg[0])) if np.linalg.norm(seg) > 0.1 else 0.0
    Lm = np.linalg.norm(np.diff(np.concatenate([np.zeros((1, 2)), p]), axis=0), axis=1).sum()
    flag = "  <- under 0.2%" if share[m] < 0.002 else ""
    print(f"{m:>4}  {GROUP_NAMES[_gown[m]]:<20}({p[-1, 0]:7.1f},{p[-1, 1]:7.1f}){Lm:>8.1f}m{hd:>8.1f}°{share[m]:>13.2%}{flag}")

fig, axs = plt.subplots(1, 2, figsize=(14, 7))
for ax, lim in zip(axs, (None, 30)):
    for m in range(NUM_MODES):
        c = GROUP_COLORS[_gown[m]]
        ax.plot(_ga[m, :, 0], _ga[m, :, 1], "-", color=c, lw=1.8)
        ax.plot(_ga[m, -1, 0], _ga[m, -1, 1], "o", color=c, ms=4)
        if lim is None or (abs(_ga[m, -1, 0]) < lim and abs(_ga[m, -1, 1]) < lim):
            ax.text(_ga[m, -1, 0], _ga[m, -1, 1], f" {m}", fontsize=8)
    handles = [plt.Line2D([], [], color=GROUP_COLORS[g], lw=3,
                          label=f"{GROUP_NAMES[g]} (slots {sum(GROUP_SLOTS[:g])}-{sum(GROUP_SLOTS[:g + 1]) - 1})")
               for g in range(len(GROUP_NAMES))]
    ax.legend(handles=handles, fontsize=8, loc="upper left")
    if lim is not None:
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel("x (m, forward)"); ax.set_ylabel("y (m, left)")
    ax.set_title("Grouped anchors" + ("" if lim is None else f" (zoom +-{lim} m)"))
plt.tight_layout(); plt.show()

In [ ]:
# Check: the GPU winner rule matches the numpy rule, and a 300-step smoke run with the current loss.

GROUP_ANCHORS = grouped_anchor_paths.to(DEVICE)
GROUP_OF_ANCHOR = ANCHOR_GROUP.to(DEVICE)

# (a) parity
_tw = []
with torch.no_grad():
    for i in range(0, len(_fit_set), 8192):
        ch = _fit_set[i:i + 8192]
        fx = torch.stack([s["future_xy"] for s in ch]).float().to(DEVICE)
        fv = torch.stack([s["future_valid"] for s in ch]).float().to(DEVICE)
        _tw.append(grouped_winner(fx, fv).cpu())
_tw = torch.cat(_tw).numpy()
print(f"(a) GPU vs numpy assigned slot agreement on {len(_tw)} futures: {(_tw == _w).mean():.4%} "
      f"({(_tw != _w).sum()} differ)")

# (b) smoke training
print(f"(b) ASSIGNMENT_MODE={ASSIGNMENT_MODE} GROUPED_HYBRID={GROUPED_HYBRID} CONF_TARGET={CONF_TARGET}")
_SMOKE_STEPS, _SMOKE_BS = 300, 128
torch.manual_seed(0)
_sm = TrajectoryTransformer(anchor_paths=grouped_anchor_paths, anchor_group=ANCHOR_GROUP,
                            num_groups=len(GROUP_NAMES) if GROUP_QUERY_EMBED else 0).to(DEVICE)
_opt = torch.optim.AdamW(_sm.parameters(), lr=3e-4, weight_decay=1e-4)
_sm.train()
_log = []
_t = time.time()
for step in range(_SMOKE_STEPS):
    lr = 3e-4 * min(1.0, (step + 1) / 100)
    for g_ in _opt.param_groups:
        g_["lr"] = lr
    batch = train_scenes[(step * _SMOKE_BS) % 200_000:(step * _SMOKE_BS) % 200_000 + _SMOKE_BS]
    f, v, fx, fv, rx, rt, rv, h = [x.to(DEVICE) for x in collate_scenes(batch)]
    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
        pt, pc, pl = _sm(f, v, rx, rt, rv, h)
    loss, nll, conf = gaussian_nll_loss(pt, pc, pl, fx, fv, use_learned_std=False)
    _opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(_sm.parameters(), 1.0)
    _opt.step()
    _log.append([float(loss.detach()), float(nll), float(conf), float(LOSS_STATS["group_top1"]),
                 float(LOSS_STATS["assigned_top1"]), float(LOSS_STATS["reg_is_assigned"]),
                 float(LOSS_STATS["tgt_entropy"])])
    if (step + 1) % 50 == 0:
        a = np.array(_log[-50:]).mean(0)
        print(f"  step {step + 1}: loss {a[0]:.3f} nll {a[1]:.3f} conf {a[2]:.3f} (target H {a[6]:.2f}) "
              f"gtop1 {a[3]:.1%} atop1 {a[4]:.1%} reg=asg {a[5]:.1%}", flush=True)
assert all(np.isfinite(np.array(_log)).ravel()), "non-finite values in the smoke log"
print(f"(b) smoke OK: {_SMOKE_STEPS} steps in {time.time() - _t:.0f} s | shapes: trajs {tuple(pt.shape)}, "
      f"confs {tuple(pc.shape)} | group embedding: {_sm.num_groups} groups")
del _sm, _opt
torch.cuda.empty_cache()

In [ ]:
# Metrics: top-6 (endpoint NMS) minADE / minFDE / miss rate, calibration, off-road rate and Waymo-style metrics.

OFFROAD_THRESH_M = 3.0
OFFROAD_STEP_STRIDE = 10   # check one point per second of the 8 s horizon
NMS_DIST_M = 2.5           # MTR's endpoint NMS threshold


def select_modes(pred_trajs, pred_confs, k, dist_thresh=NMS_DIST_M):
    """pred_trajs [B, M, T, 2], pred_confs [B, M] -> (idx [B, k], probs [B, k]), most confident first.
    Endpoint NMS: keep a mode unless its final point is within dist_thresh of an already-kept mode's
    final point; if fewer than k survive, fill with the most confident suppressed modes.
    idx[:, 0] is always the single most confident mode."""
    B, M = pred_confs.shape
    k = min(k, M)
    dev = pred_confs.device
    probs = F.softmax(pred_confs, dim=-1)                                              # [B, M]
    order = probs.argsort(dim=1, descending=True)                                      # [B, M]
    ends = torch.gather(pred_trajs[:, :, -1], 1, order.unsqueeze(-1).expand(B, M, 2))  # [B, M, 2], sorted
    close = torch.cdist(ends, ends) < dist_thresh                                      # [B, M, M]
    kept = torch.zeros(B, M, dtype=torch.bool, device=dev)
    n_kept = torch.zeros(B, dtype=torch.long, device=dev)
    for j in range(M):
        suppressed = (close[:, j] & kept).any(dim=1)
        take = ~suppressed & (n_kept < k)
        kept[:, j] = take
        n_kept += take.long()
    rank_key = (~kept).long() * M + torch.arange(M, device=dev)    # kept modes first, each group by confidence
    pos = rank_key.argsort(dim=1)[:, :k]
    idx = torch.gather(order, 1, pos)
    return idx, torch.gather(probs, 1, idx)


def _offroad_flags(traj, traj_valid, lane_xy, lane_mask, reach, center):
    """traj [B, K, S, 2] m, traj_valid [B, K, S] bool, lane_xy [B, R, 2] m, lane_mask [B, R] bool,
    reach [B] m -> [B, K] bool: any checkable point farther than the threshold from every lane point."""
    B, K, S, _ = traj.shape
    d = torch.cdist(traj.reshape(B, K * S, 2), lane_xy)                       # [B, K*S, R]
    d = d.masked_fill(~lane_mask.unsqueeze(1), float("inf")).min(dim=-1).values.view(B, K, S)
    checkable = traj_valid & ((traj - center).norm(dim=-1) <= reach.view(B, 1, 1))
    return ((d > OFFROAD_THRESH_M) & checkable).any(dim=-1)


def compute_metrics(model, scenes, batch_size=128, miss_threshold_m=2.0, return_per_sample=False, top_k=None):
    top_k = EVAL_TOP_K if top_k is None else top_k
    model.eval()
    min_ades, min_fdes, misses, p_top1s, top1_best = [], [], [], [], []
    off_modes, off_top1, off_gt = [], [], []
    center = torch.tensor([ROAD_FORWARD_OFFSET_M, 0.0], device=DEVICE)   # crop center in the agent frame

    with torch.no_grad():
        for i in range(0, len(scenes), batch_size):
            batch = scenes[i : i + batch_size]
            feats, valid, future_xy, future_valid, road_xy, road_type, road_valid, heading = collate_scenes(batch)
            feats, valid = feats.to(DEVICE), valid.to(DEVICE)
            future_xy, future_valid = future_xy.to(DEVICE), future_valid.to(DEVICE)
            road_xy, road_type, road_valid = road_xy.to(DEVICE), road_type.to(DEVICE), road_valid.to(DEVICE)
            heading = heading.to(DEVICE)

            pred_trajs, pred_confs, _ = model(feats, valid, road_xy, road_type, road_valid, heading)  # [B, M, T, 2]
            B, M, T, _ = pred_trajs.shape
            k = min(top_k, M)
            top_idx, top_p = select_modes(pred_trajs, pred_confs, k)       # [B, k], most confident first
            pred_trajs = torch.gather(pred_trajs, 1, top_idx[:, :, None, None].expand(B, k, T, 2))  # [B, k, T, 2]

            dist = (pred_trajs - future_xy.unsqueeze(1)).norm(dim=-1)  # [B, k, T]
            valid_exp = future_valid.unsqueeze(1)                      # [B, 1, T]

            ade_per_mode = (dist * valid_exp).sum(-1) / valid_exp.sum(-1).clamp(min=1.0)  # [B, k]
            min_ade = ade_per_mode.min(dim=1).values

            # last valid step (validity is not always a contiguous prefix)
            time_idx = torch.arange(future_valid.shape[1], device=future_valid.device)
            last_idx = (future_valid * time_idx.unsqueeze(0)).max(dim=1).values.long()
            fde_per_mode = dist[torch.arange(B, device=dist.device), :, last_idx]  # [B, k]
            min_fde = fde_per_mode.min(dim=1).values

            min_ades.extend(min_ade.cpu().tolist())
            min_fdes.extend(min_fde.cpu().tolist())
            misses.extend((min_fde > miss_threshold_m).float().cpu().tolist())
            p_top1s.extend(top_p[:, 0].cpu().tolist())
            top1_best.extend((ade_per_mode.argmin(dim=1) == 0).float().cpu().tolist())

            # Off-road check on vehicle targets with lane centers in the crop
            map_xy = road_xy[:, :MAX_ROAD_POINTS, :2] * POS_SCALE_M                  # [B, R, 2] m
            map_valid = road_valid[:, :MAX_ROAD_POINTS] > 0
            lane_mask = map_valid & (road_type[:, :MAX_ROAD_POINTS] == 1)            # bucket 1 = lane center
            reach = torch.where(map_valid, (map_xy - center).norm(dim=-1),
                                torch.zeros_like(map_valid, dtype=map_xy.dtype)).max(dim=1).values  # [B]
            eligible = (feats[:, 0, -1, 6] > 0) & lane_mask.any(dim=1)               # vehicles with lanes

            sidx = torch.arange(OFFROAD_STEP_STRIDE - 1, T, OFFROAD_STEP_STRIDE, device=DEVICE)
            pred_s = pred_trajs[:, :, sidx]                                          # [B, k, S, 2]
            pred_off = _offroad_flags(pred_s, torch.ones(pred_s.shape[:3], dtype=torch.bool, device=DEVICE),
                                      map_xy, lane_mask, reach, center)              # [B, k]
            gt_s = future_xy[:, sidx].unsqueeze(1)                                   # [B, 1, S, 2]
            gt_off = _offroad_flags(gt_s, (future_valid[:, sidx] > 0).unsqueeze(1),
                                    map_xy, lane_mask, reach, center)[:, 0]          # [B]

            off_modes.extend(pred_off.float().mean(dim=1)[eligible].cpu().tolist())
            off_top1.extend(pred_off[:, 0].float()[eligible].cpu().tolist())
            off_gt.extend(gt_off.float()[eligible].cpu().tolist())

    model.train()
    result = {
        "minADE": float(np.mean(min_ades)),
        "minFDE": float(np.mean(min_fdes)),
        "miss_rate": float(np.mean(misses)),
        "p_top1": float(np.mean(p_top1s)),
        "top1_is_best": float(np.mean(top1_best)),
        "offroad_rate": float(np.mean(off_modes)) if off_modes else float("nan"),       # all selected modes
        "offroad_top1": float(np.mean(off_top1)) if off_top1 else float("nan"),         # most confident mode
        "gt_offroad_rate": float(np.mean(off_gt)) if off_gt else float("nan"),          # baseline
    }
    if return_per_sample:
        result["per_sample"] = {"minADE": np.array(min_ades), "minFDE": np.array(min_fdes),
                                "miss": np.array(misses)}
    return result


# Waymo-style metrics: 3 / 5 / 8 s on 2 Hz points, speed-scaled lateral / longitudinal miss boxes (no mAP)

OFFICIAL_HORIZONS = ((3, 29, 1.0, 2.0), (5, 49, 1.8, 3.6), (8, 79, 3.0, 6.0))   # (s, step idx, lat, lon)
OFFICIAL_STEPS_2HZ = torch.arange(4, FUTURE_STEPS, 5)                           # 0.5 s, 1.0 s, ... 8.0 s


def _gt_heading(gt, gv, e, lag=5):
    """Unit vector of the ground truth's direction of travel over [e - lag, e]; +x if it barely moved."""
    s = max(e - lag, 0)
    d = gt[:, e] - gt[:, s]                                                   # [B, 2]
    ok = (gv[:, e] > 0) & (gv[:, s] > 0) & (d.norm(dim=-1) > 0.5)
    fallback = torch.tensor([1.0, 0.0], device=gt.device).expand_as(d)
    return torch.where(ok.unsqueeze(-1), d / d.norm(dim=-1, keepdim=True).clamp(min=1e-6), fallback)


def compute_official_metrics(model, scenes, batch_size=256, top_k=None):
    top_k = EVAL_TOP_K if top_k is None else top_k
    model.eval()
    acc = {f"{n}@{h}s": [] for h, *_ in OFFICIAL_HORIZONS for n in ("minADE", "minFDE", "MR")}
    types = []
    steps2 = OFFICIAL_STEPS_2HZ.to(DEVICE)
    with torch.no_grad():
        for i in range(0, len(scenes), batch_size):
            feats, valid, gt, gv, road_xy, road_type, road_valid, heading = \
                [t.to(DEVICE) for t in collate_scenes(scenes[i:i + batch_size])]
            pred, conf, _ = model(feats, valid, road_xy, road_type, road_valid, heading)
            B, M, T, _ = pred.shape
            k = min(top_k, M)
            idx, _ = select_modes(pred, conf, k)
            pred = torch.gather(pred, 1, idx[:, :, None, None].expand(B, k, T, 2))   # [B, k, T, 2]
            err = pred - gt.unsqueeze(1)                                            # [B, k, T, 2]
            dist = err.norm(dim=-1)                                                 # [B, k, T]
            speed0 = feats[:, 0, -1, 2:4].norm(dim=-1) * VELOCITY_SCALE_MPS          # [B] m/s now
            scale = (0.5 + 0.5 * (speed0 - 1.4) / (11.0 - 1.4)).clamp(0.5, 1.0)     # [B]
            nan = torch.full((B,), float("nan"), device=DEVICE)
            for h_s, e, lat_t, lon_t in OFFICIAL_HORIZONS:
                pts = steps2[steps2 <= e]
                pv = gv[:, pts]                                                     # [B, P]
                n_pts = pv.sum(-1)
                ade = (dist[:, :, pts] * pv.unsqueeze(1)).sum(-1) / n_pts.clamp(min=1).unsqueeze(1)   # [B, k]
                ve = gv[:, e] > 0
                hvec = _gt_heading(gt, gv, e)                                       # [B, 2] longitudinal axis
                nvec = torch.stack([-hvec[:, 1], hvec[:, 0]], dim=-1)               # [B, 2] lateral axis
                e_err = err[:, :, e]                                                # [B, k, 2]
                lon = (e_err * hvec.unsqueeze(1)).sum(-1).abs()
                lat = (e_err * nvec.unsqueeze(1)).sum(-1).abs()
                match = (lat <= lat_t * scale[:, None]) & (lon <= lon_t * scale[:, None])   # [B, k]
                acc[f"minADE@{h_s}s"].append(torch.where(n_pts > 0, ade.min(1).values, nan).cpu())
                acc[f"minFDE@{h_s}s"].append(torch.where(ve, dist[:, :, e].min(1).values, nan).cpu())
                acc[f"MR@{h_s}s"].append(torch.where(ve, (~match.any(1)).float(), nan).cpu())
            oh = feats[:, 0, -1, 6:9]
            types.append(torch.where(oh.sum(-1) > 0, oh.argmax(-1), torch.full_like(oh[:, 0], -1).long()).cpu())
    model.train()

    per = {k: torch.cat(v).numpy() for k, v in acc.items()}
    types = torch.cat(types).numpy()
    res = {k: float(np.nanmean(v)) for k, v in per.items()}
    for n in ("minADE", "minFDE", "MR"):
        res[f"{n}_avg"] = float(np.mean([res[f"{n}@{h}s"] for h, *_ in OFFICIAL_HORIZONS]))
    res["per_type"] = {}
    for t, tn in enumerate(["vehicle", "pedestrian", "cyclist"]):
        m = types == t
        if m.any():
            res["per_type"][tn] = {"n": int(m.sum()), **{
                f"{n}_avg": float(np.mean([np.nanmean(per[f"{n}@{h}s"][m]) for h, *_ in OFFICIAL_HORIZONS]))
                for n in ("minADE", "minFDE", "MR")}}
    return res


In [ ]:
# Training: time-boxed AdamW with warmup + cosine LR, GPU-resident data, resumable checkpoints, best checkpoint by val minADE.

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

RESUME_RUN_DIR = None   # e.g. "/content/drive/MyDrive/waymo_trajectory/run_20261002_1030" to resume
if RESUME_RUN_DIR:
    CKPT_DIR = RESUME_RUN_DIR
    RUN_NAME = os.path.basename(RESUME_RUN_DIR.rstrip("/"))
else:
    RUN_NAME = time.strftime("run_%Y%m%d_%H%M")
    CKPT_DIR = f"/content/drive/MyDrive/waymo_trajectory/{RUN_NAME}"
os.makedirs(CKPT_DIR, exist_ok=True)
CHECKPOINT_PATH = f"{CKPT_DIR}/trajectory_transformer_checkpoint.pt"   # periodic, resumable
BEST_CHECKPOINT_PATH = f"{CKPT_DIR}/trajectory_transformer_best.pt"    # best val minADE

TRAIN_TIME_BUDGET_SEC = 10 * 60 * 60    # total across resumes
CHECKPOINT_EVERY_SEC = 10 * 60
LOG_EVERY_STEPS = 1000

BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
PEAK_LR = 3e-4          # peak learning rate
MIN_LR = 1e-6
WARMUP_STEPS = 3_000
NLL_WARMUP_STEPS = 15_000   # sigma held at 1 until here
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
ORACLE_VAL_N = 20_000       # val subset for the best-of-all-modes diagnostic

USE_BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for remaining fp32 matmuls (Ampere+)
torch.backends.cudnn.allow_tf32 = True


def lr_schedule(step, elapsed_sec):
    """Linear warmup over WARMUP_STEPS, times a cosine decay from PEAK_LR to MIN_LR over the time budget."""
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    progress = min(1.0, elapsed_sec / TRAIN_TIME_BUDGET_SEC)
    return warm * (MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress)))


# ---- data on the GPU ----
_BANK_KEYS = ["agent_feats", "agent_valid", "future_xy", "future_valid", "road_xy", "road_type", "road_valid"]


def make_gpu_bank(scenes, chunk=100_000):
    bank = {}
    for k in _BANK_KEYS:
        parts = [torch.stack([s[k] for s in scenes[i:i + chunk]]).to(DEVICE) for i in range(0, len(scenes), chunk)]
        bank[k] = torch.cat(parts)
        del parts
    return bank


_t0 = time.time()
if globals().get("TRAIN_BANK_N") != len(train_scenes):
    TRAIN_BANK = make_gpu_bank(train_scenes); TRAIN_BANK_N = len(train_scenes)
if globals().get("VAL_BANK_N") != len(val_scenes):
    VAL_BANK = make_gpu_bank(val_scenes); VAL_BANK_N = len(val_scenes)
torch.cuda.synchronize()
print(f"GPU banks ready in {(time.time() - _t0) / 60:.1f} min: train {TRAIN_BANK_N}, val {VAL_BANK_N} | "
      f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.1f} GB")

_HEADING = torch.tensor([1.0, 0.0], device=DEVICE)


def bank_batch(bank, idx):
    """Same 8-tuple as collate_scenes, sliced from a GPU bank and cast to the model's dtypes."""
    return (bank["agent_feats"][idx].float(), bank["agent_valid"][idx].float(),
            bank["future_xy"][idx].float(), bank["future_valid"][idx].float(),
            bank["road_xy"][idx].float(), bank["road_type"][idx].long(),
            bank["road_valid"][idx].float(), _HEADING.expand(len(idx), 2))


def compute_val_loss(model, bank, use_learned_std=True, batch_size=EVAL_BATCH_SIZE):
    """Training loss on held-out samples (reported only; selection uses val minADE).
    Returns (loss, rank_top1); rank_top1 is nan unless ASSIGNMENT_MODE == "grouped"."""
    model.eval()
    total, nb = torch.zeros((), device=DEVICE), 0
    atop1 = torch.zeros((), device=DEVICE)
    n = bank["agent_feats"].shape[0]
    with torch.no_grad():
        for i in range(0, n - batch_size + 1, batch_size):
            idx = torch.arange(i, i + batch_size, device=DEVICE)
            f, v, fx, fv, rx, rt, rv, h = bank_batch(bank, idx)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
                pt, pc, pl = model(f, v, rx, rt, rv, h)
            loss, _, _ = gaussian_nll_loss(pt, pc, pl, fx, fv, use_learned_std=use_learned_std)
            total += loss
            atop1 += LOSS_STATS.get("rank_top1", torch.full((), float("nan"), device=DEVICE))
            nb += 1
    model.train()
    return float(total) / max(nb, 1), float(atop1) / max(nb, 1)


def oracle_minade(model, bank, n=ORACLE_VAL_N, batch_size=EVAL_BATCH_SIZE):
    """Best-of-ALL-modes minADE on the first n samples: regression quality, independent of ranking."""
    model.eval()
    total = torch.zeros((), device=DEVICE)
    n = min(n, bank["agent_feats"].shape[0])
    with torch.no_grad():
        for i in range(0, n, batch_size):
            idx = torch.arange(i, min(i + batch_size, n), device=DEVICE)
            f, v, fx, fv, rx, rt, rv, h = bank_batch(bank, idx)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
                pt, _, _ = model(f, v, rx, rt, rv, h)
            w = fv[:, None]
            ade = ((pt - fx[:, None]).norm(dim=-1) * w).sum(-1) / w.sum(-1).clamp(min=1)
            total += ade.min(1).values.sum()
    model.train()
    return float(total) / n


_conf_desc = {"relative": f"relative tau = {REL_TAU_SCALE}*ade_min + {REL_TAU_BIAS} (max {REL_TAU_MAX}) m",
              "fixed": f"fixed tau = {CONF_TARGET_TEMP_M} m",
              "hard": f"hard (label smoothing {LABEL_SMOOTHING})"}[CONF_TARGET]
if ASSIGNMENT_MODE == "grouped" and GROUPED_HYBRID:
    _conf_desc = (f"grouped hybrid: GT group fixed, in-group relative tau = {REL_TAU_SCALE}*ade_min + "
                  f"{REL_TAU_BIAS} (max {REL_TAU_MAX}) m; regression = closest slot in the GT group")

RUN_CONFIG = dict(
    run_name=RUN_NAME, seed=SEED, prepped_dir=PREP_OUT_DIR, anchor_space=ANCHOR_SPACE,
    assignment_mode=ASSIGNMENT_MODE, conf_target=CONF_TARGET, rel_tau_scale=REL_TAU_SCALE,
    rel_tau_bias=REL_TAU_BIAS, rel_tau_max=REL_TAU_MAX, soft_conf_target=SOFT_CONF_TARGET,
    conf_target_temp_m=CONF_TARGET_TEMP_M, label_smoothing=LABEL_SMOOTHING,
    nll_weight=NLL_WEIGHT, conf_weight=CONF_WEIGHT,
    cumulative_offsets=CUMULATIVE_OFFSETS, EVAL_TOP_K=EVAL_TOP_K, NMS_DIST_M=NMS_DIST_M,
    batch_size=BATCH_SIZE, peak_lr=PEAK_LR, min_lr=MIN_LR, warmup_steps=WARMUP_STEPS,
    nll_warmup_steps=NLL_WARMUP_STEPS, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP_NORM,
    lr_schedule="warmup + cosine over time budget", time_budget_sec=TRAIN_TIME_BUDGET_SEC, use_bf16=USE_BF16,
    MAX_AGENTS=MAX_AGENTS, HIST_STEPS=HIST_STEPS, FUTURE_STEPS=FUTURE_STEPS,
    MAX_ROAD_POINTS=MAX_ROAD_POINTS, MAX_TRAFFIC_LIGHTS=MAX_TRAFFIC_LIGHTS, ROAD_RADIUS_M=ROAD_RADIUS_M,
    ROAD_POINT_STRIDE=ROAD_POINT_STRIDE, ROAD_FORWARD_OFFSET_M=ROAD_FORWARD_OFFSET_M,
    POS_SCALE_M=POS_SCALE_M, VELOCITY_SCALE_MPS=VELOCITY_SCALE_MPS,
    AGENT_FEAT_DIM=AGENT_FEAT_DIM, ROAD_FEAT_DIM=ROAD_FEAT_DIM, NUM_ROAD_BUCKETS=NUM_ROAD_BUCKETS,
    TL_BUCKET_OFFSET=TL_BUCKET_OFFSET, NUM_MODES=NUM_MODES, D_MODEL=D_MODEL, N_HEADS=N_HEADS,
    N_LAYERS=N_LAYERS, DECODER_ROUNDS=DECODER_ROUNDS, FF_DIM=FF_DIM, DECODER_FF_DIM=DECODER_FF_DIM,
    DROPOUT=DROPOUT, LOG_STD_MIN=LOG_STD_MIN, LOG_STD_MAX=LOG_STD_MAX, anchor_conditioned_queries=True,
    split=(f"by shard id%10: 8 = val, 9 = test, else train "
           f"({n_train_shards} / {n_val_shards} / {n_test_shards} shards)"),
    n_train=TRAIN_BANK_N, n_val=VAL_BANK_N, n_test=len(test_scenes),
)

# ---- grouped anchors ----
_model_kw = {}
if ASSIGNMENT_MODE == "grouped":
    if "grouped_anchor_paths" not in globals():
        raise RuntimeError("ASSIGNMENT_MODE = 'grouped' needs the maneuver-group and grouped-anchor cells run first.")
    anchor_paths = grouped_anchor_paths            # the bank this run trains on (saved in checkpoints)
    GROUP_ANCHORS = anchor_paths.to(DEVICE)
    GROUP_OF_ANCHOR = ANCHOR_GROUP.to(DEVICE)
    _model_kw = dict(anchor_group=ANCHOR_GROUP, num_groups=len(GROUP_NAMES) if GROUP_QUERY_EMBED else 0)
    RUN_CONFIG.update(anchor_bank="grouped", group_names=GROUP_NAMES, group_slots=GROUP_SLOTS,
                      group_kmeans_standardize=GROUP_KMEANS_STANDARDIZE, turn_deg=TURN_DEG, uturn_deg=UTURN_DEG,
                      uturn_min_lat_m=UTURN_MIN_LAT_M, stationary_len_m=STATIONARY_LEN_M,
                      group_query_embed=GROUP_QUERY_EMBED, anchor_group=ANCHOR_GROUP.tolist(),
                      grouped_hybrid=GROUPED_HYBRID)
    print(f"Grouped anchors: slots per group {dict(zip(GROUP_NAMES, GROUP_SLOTS))}, "
          f"group query embedding {'on' if GROUP_QUERY_EMBED else 'off'}")

# ---- model, optimizer, resume ----
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model = TrajectoryTransformer(anchor_paths=anchor_paths, **_model_kw).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=WEIGHT_DECAY)
state = dict(epoch=0, batch_pos=0, global_step=0, elapsed=0.0, best_val_minade=float("inf"))

if RESUME_RUN_DIR and os.path.exists(CHECKPOINT_PATH):
    ck = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ck["model_state_dict"])          # includes the run's anchor buffer
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    anchor_paths = ck["anchor_paths"]
    ANCHOR_KF = {k: v.to(DEVICE) for k, v in ck["anchor_kf"].items()}
    if ASSIGNMENT_MODE == "grouped":
        GROUP_ANCHORS = anchor_paths.to(DEVICE)
        GROUP_OF_ANCHOR = torch.as_tensor(ck["anchor_group"]).long().to(DEVICE)
    state.update(ck["train_state"])
    print(f"Resumed {RUN_NAME}: epoch {state['epoch']} + {state['batch_pos']} batches, step {state['global_step']}, "
          f"{state['elapsed'] / 3600:.2f} h elapsed, best val minADE {state['best_val_minade']:.3f}")
else:
    print(f"Fresh init (seed {SEED}).")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | bf16 autocast: {USE_BF16} | "
      f"cumulative offsets: {model.cumulative_offsets} | anchors: {ANCHOR_SPACE}")


def _save(path, with_optimizer=False, extra=None):
    payload = {"model_state_dict": model.state_dict(), "anchor_paths": anchor_paths,
               "anchor_kf": {k: v.detach().cpu() for k, v in ANCHOR_KF.items()},
               "config": RUN_CONFIG, "train_state": dict(state),
               "anchor_group": GROUP_OF_ANCHOR.detach().cpu() if ASSIGNMENT_MODE == "grouped" else None,
               "epoch": state["epoch"], "loss": extra.get("train_loss", float("nan")) if extra else float("nan")}
    if with_optimizer:
        payload["optimizer_state_dict"] = optimizer.state_dict()
    if extra:
        payload.update(extra)
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)     # atomic: a crash mid-save never leaves a corrupt checkpoint


def validate(tag, use_learned_std, train_loss):
    val_m = compute_metrics(model, val_scenes, batch_size=EVAL_BATCH_SIZE)
    val_loss, val_atop1 = compute_val_loss(model, VAL_BANK, use_learned_std=use_learned_std)
    oracle = oracle_minade(model, VAL_BANK)
    improved = val_m["minADE"] < state["best_val_minade"]
    if improved:
        state["best_val_minade"] = val_m["minADE"]
        _save(BEST_CHECKPOINT_PATH, extra={
            "train_loss": train_loss, "val_minADE": val_m["minADE"], "val_minFDE": val_m["minFDE"],
            "val_miss_rate": val_m["miss_rate"], "val_loss": val_loss, "val_oracle_minADE_20k": oracle,
            "val_p_top1": val_m["p_top1"], "val_top1_is_best": val_m["top1_is_best"],
            "val_offroad_rate": val_m["offroad_rate"], "val_gt_offroad_rate": val_m["gt_offroad_rate"],
            "val_assigned_top1": val_atop1})
    print(f"  VAL {tag}: minADE {val_m['minADE']:.3f} (best of {NUM_MODES}: {oracle:.3f}) "
          f"minFDE {val_m['minFDE']:.2f} MR {val_m['miss_rate']:.1%} "
          f"p_top1 {val_m['p_top1']:.2f} top1_is_best {val_m['top1_is_best']:.1%} "
          f"off-road {val_m['offroad_rate']:.1%} (GT {val_m['gt_offroad_rate']:.1%}) "
          f"{_RANK_LABEL} {val_atop1:.1%} val_loss {val_loss:.4f}{'  (best)' if improved else ''}", flush=True)


# ---- training loop ----
_HYBRID_ON = ASSIGNMENT_MODE == "grouped" and GROUPED_HYBRID
_RANK_LABEL = "gtop1" if _HYBRID_ON else "atop1"
_REG_TXT = (lambda x: f" reg=asg {x:.1%}") if _HYBRID_ON else (lambda x: "")
N_TRAIN = TRAIN_BANK_N
STEPS_PER_EPOCH = N_TRAIN // BATCH_SIZE     # last partial batch dropped
start_wall = time.time() - state["elapsed"]
elapsed = lambda: time.time() - start_wall
last_ckpt = time.time()
interrupted = False
print(f"Training up to {TRAIN_TIME_BUDGET_SEC / 3600:.1f} h | {N_TRAIN} train samples, {STEPS_PER_EPOCH} steps/epoch "
      f"at batch {BATCH_SIZE} | LR warmup {WARMUP_STEPS} steps to {PEAK_LR:.0e}, cosine to {MIN_LR:.0e} over the budget | "
      f"sigma=1 for {NLL_WARMUP_STEPS} steps | conf target: {_conf_desc} | "
      f"loss = {NLL_WEIGHT}*nll + {CONF_WEIGHT}*conf | checkpoints -> {CKPT_DIR}", flush=True)

model.train()
try:
    while elapsed() < TRAIN_TIME_BUDGET_SEC:
        epoch = state["epoch"] + 1
        perm = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(SEED + epoch)).to(DEVICE)
        ep_sum = torch.zeros(3, device=DEVICE); ep_n = 0
        # window: loss, nll, conf, p_top1, KL, target entropy, target winner mass, rank top-1, reg==assigned
        win = torch.zeros(9, device=DEVICE); win_n = 0; win_t = time.time()
        win_gn = torch.zeros((), device=DEVICE)

        for b in range(state["batch_pos"], STEPS_PER_EPOCH):
            if elapsed() >= TRAIN_TIME_BUDGET_SEC:
                break
            idx = perm[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            f, v, fx, fv, rx, rt, rv, h = bank_batch(TRAIN_BANK, idx)
            use_learned_std = state["global_step"] >= NLL_WARMUP_STEPS

            lr = lr_schedule(state["global_step"], elapsed())
            for g in optimizer.param_groups:
                g["lr"] = lr

            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
                pt, pc, pl = model(f, v, rx, rt, rv, h)
            loss, nll, conf = gaussian_nll_loss(pt, pc, pl, fx, fv, use_learned_std=use_learned_std)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            state["global_step"] += 1
            state["batch_pos"] = b + 1
            ld = loss.detach()
            ep_sum += torch.stack([ld, nll, conf]); ep_n += 1
            win += torch.stack([ld, nll, conf, LOSS_STATS["p_top1"], conf - LOSS_STATS["tgt_entropy"],
                                LOSS_STATS["tgt_entropy"], LOSS_STATS["tgt_win"],
                                LOSS_STATS.get("rank_top1", torch.zeros((), device=DEVICE)),
                                LOSS_STATS.get("reg_is_assigned", torch.zeros((), device=DEVICE))]); win_n += 1
            win_gn += gn.detach()

            if state["global_step"] % LOG_EVERY_STEPS == 0:
                w = (win / win_n).tolist()
                if not math.isfinite(w[0]):
                    raise RuntimeError(f"Non-finite loss near step {state['global_step']}: set USE_BF16 = False and resume.")
                sps = win_n * BATCH_SIZE / (time.time() - win_t)
                proj = TRAIN_TIME_BUDGET_SEC * sps / N_TRAIN
                print(f"step {state['global_step']} (ep {epoch} {b + 1}/{STEPS_PER_EPOCH}, {elapsed() / 60:.1f} min) "
                      f"loss {w[0]:.3f} nll {w[1]:.3f} conf {w[2]:.3f} (KL {w[4]:.3f}, target H {w[5]:.2f} win {w[6]:.2f}) "
                      f"p_top1 {w[3]:.2f} {_RANK_LABEL} {w[7]:.1%}{_REG_TXT(w[8])} grad {float(win_gn) / win_n:.2f} lr {lr:.2e} | {sps:.0f} samples/s, "
                      f"~{proj:.1f} epochs in budget{'' if use_learned_std else ' [sigma=1]'}", flush=True)
                win.zero_(); win_gn.zero_(); win_n = 0; win_t = time.time()

            if time.time() - last_ckpt >= CHECKPOINT_EVERY_SEC:
                state["elapsed"] = elapsed()
                _save(CHECKPOINT_PATH, with_optimizer=True)
                last_ckpt = time.time()
        else:
            # full epoch completed
            state["epoch"] = epoch
            state["batch_pos"] = 0
            state["elapsed"] = elapsed()
            e = (ep_sum / max(ep_n, 1)).tolist()
            print(f"Epoch {epoch} done ({elapsed() / 60:.1f} min): loss {e[0]:.4f} nll {e[1]:.4f} conf {e[2]:.4f}", flush=True)
            validate(f"epoch {epoch}", state["global_step"] >= NLL_WARMUP_STEPS, e[0])
            _save(CHECKPOINT_PATH, with_optimizer=True)
            last_ckpt = time.time()
            continue
        break   # inner loop broke: time budget reached mid-epoch
except KeyboardInterrupt:
    interrupted = True
    print("\nInterrupted.")

state["elapsed"] = elapsed()
_save(CHECKPOINT_PATH, with_optimizer=True)
if not interrupted and state["batch_pos"] > 0:
    validate(f"final partial epoch {state['epoch'] + 1} ({state['batch_pos']}/{STEPS_PER_EPOCH} batches)",
             state["global_step"] >= NLL_WARMUP_STEPS, float("nan"))

print(f"\nDone{' (interrupted -- set RESUME_RUN_DIR to continue)' if interrupted else ''}: "
      f"{state['global_step']} steps, {state['epoch']} full epochs, {state['elapsed'] / 60:.1f} min, "
      f"best val minADE {state['best_val_minade']:.3f} m")
print(f"Best checkpoint (by val minADE): {BEST_CHECKPOINT_PATH}")


In [ ]:
# Load the best checkpoint with its own anchors, head type and group settings (LOAD_RUN_DIR selects a run).

LOAD_RUN_DIR = None

_run_dir = LOAD_RUN_DIR or globals().get("CKPT_DIR") or "/content/drive/MyDrive/waymo_trajectory/run_20261004_2029"
BEST_CHECKPOINT_PATH = f"{_run_dir}/trajectory_transformer_best.pt"
print(f"Loading {BEST_CHECKPOINT_PATH}")

ckpt = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE)
ckpt_cfg = ckpt.get("config", {})
_conf_head_type = "linear" if "conf_head.weight" in ckpt["model_state_dict"] else "mlp"
_sd = ckpt["model_state_dict"]
_num_groups = _sd["group_embed.weight"].shape[0] if "group_embed.weight" in _sd else 0
_ckpt_group = ckpt.get("anchor_group")
if _ckpt_group is None and "anchor_group" in _sd:
    _ckpt_group = _sd["anchor_group"]

# anchors from the checkpoint (fallback: this session's anchors)
ckpt_anchors = ckpt.get("anchor_paths")
if ckpt_anchors is None:
    ckpt_anchors = anchor_paths
    print("WARNING: checkpoint has no anchor_paths; using refitted anchors.")

model = TrajectoryTransformer(anchor_paths=ckpt_anchors,
                              cumulative_offsets=ckpt_cfg.get("cumulative_offsets", False),
                              conf_head_type=_conf_head_type,
                              anchor_group=_ckpt_group, num_groups=_num_groups).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.set_anchors(ckpt_anchors.to(DEVICE))
model.to(DEVICE)

# restore the k-means normalizer for anchor-based assignment
if ckpt.get("anchor_kf") is not None:
    ANCHOR_KF = {k: v.to(DEVICE) for k, v in ckpt["anchor_kf"].items()}

# grouped runs: restore the bank the grouped loss uses
_ckpt_grouped = ckpt_cfg.get("assignment_mode") == "grouped"
if _ckpt_grouped:
    GROUP_ANCHORS = ckpt_anchors.to(DEVICE)
    GROUP_OF_ANCHOR = torch.as_tensor(_ckpt_group).long().to(DEVICE)

# check that the checkpoint's anchors match this session's
try:
    _session = grouped_anchor_paths if _ckpt_grouped else anchor_paths
    _match = torch.allclose(ckpt_anchors.cpu(), _session.cpu(), atol=1e-4)
    _maxdiff = (ckpt_anchors.cpu() - _session.cpu()).abs().max().item()
    print(f"Session anchors match checkpoint anchors: {_match} (max abs diff {_maxdiff:.4g})")
except Exception as e:
    print(f"(anchor comparison skipped: {e})")

print(f"Loaded best checkpoint: epoch {ckpt['epoch']}, train loss {ckpt.get('loss', float('nan')):.4f}, "
      f"val minADE {ckpt.get('val_minADE', float('nan')):.2f}m, val minFDE {ckpt.get('val_minFDE', float('nan')):.2f}m, "
      f"val_loss {ckpt.get('val_loss', float('nan')):.4f} | cumulative offsets: {model.cumulative_offsets} | "
      f"conf head: {model.conf_head_type} | anchors: {'grouped' if _ckpt_grouped else ckpt_cfg.get('anchor_space', '?')}, "
      f"group query embedding: {_num_groups > 0}")

In [ ]:
# Final evaluation on val and the held-out test split, overall and per agent type, plus Waymo-style metrics.

TYPE_SLOT_NAMES = ["vehicle", "pedestrian", "cyclist"]

def _target_type(s):
    onehot = s["agent_feats"][0, -1, 6:9].float().numpy()
    return int(np.argmax(onehot)) if onehot.sum() > 0 else -1

for name, scenes in [("VAL (used for selection)", val_scenes), ("TEST (held out)", test_scenes)]:
    m = compute_metrics(model, scenes, return_per_sample=True)
    print(f"{name}: {len(scenes)} samples | minADE {m['minADE']:.2f} m | minFDE {m['minFDE']:.2f} m "
          f"| miss rate {m['miss_rate']:.1%} (2.0 m at 8s) | top-{EVAL_TOP_K} of {NUM_MODES} modes")
    print(f"    calibration: mean top-1 confidence {m['p_top1']:.2f}, "
          f"top-1 mode is the best mode {m['top1_is_best']:.1%}")
    print(f"    off-road (vehicles, >{OFFROAD_THRESH_M:.0f} m from lane centers): top-{EVAL_TOP_K} modes "
          f"{m['offroad_rate']:.1%}, top-1 mode {m['offroad_top1']:.1%}, ground truth baseline {m['gt_offroad_rate']:.1%}")
    types = np.array([_target_type(s) for s in scenes])
    ps = m["per_sample"]
    for t, tn in enumerate(TYPE_SLOT_NAMES):
        mask = types == t
        if mask.sum():
            print(f"    {tn:<10} n={mask.sum():>6}  minADE {ps['minADE'][mask].mean():.2f}  "
                  f"minFDE {ps['minFDE'][mask].mean():.2f}  MR {ps['miss'][mask].mean():.1%}")

    o = compute_official_metrics(model, scenes)
    print("    Waymo-style (2 Hz points, speed-scaled lat/lon miss rate; heading from GT direction of travel):")
    for h, *_ in OFFICIAL_HORIZONS:
        print(f"      {h} s   minADE {o[f'minADE@{h}s']:.2f}  minFDE {o[f'minFDE@{h}s']:.2f}  MR {o[f'MR@{h}s']:.1%}")
    print(f"      avg   minADE {o['minADE_avg']:.2f}  minFDE {o['minFDE_avg']:.2f}  MR {o['MR_avg']:.1%}")
    for tn, d in o["per_type"].items():
        print(f"      {tn:<10} n={d['n']:>6}  avg minADE {d['minADE_avg']:.2f}  "
              f"minFDE {d['minFDE_avg']:.2f}  MR {d['MR_avg']:.1%}")

In [ ]:
# Prediction plots on test scenes (turn-like, random, worst): top-1, scored top-6 and all 32 modes; saved to <run>/plots/.

import os, re
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon

VIZ_ROAD_RADIUS_M = 150.0
VIZ_MAX_ROAD_POINTS = 2000
N_VIZ_SAMPLES = 20     # turn-like scenes
N_VIZ_RANDOM = 20
N_VIZ_WORST = 20
PLOT_SHARD_IDS = [9, 19, 39, 89, 99, 109, 119, 159]   # held-out test shards to draw scenes from
VIEW_MARGIN_M = 10.0       # padding around the framed content
VIEW_MIN_HALF_M = 20.0     # never zoom in tighter than +-20 m

# approximate footprint (length, width) per agent type
CAR_SIZES = {0: (4.5, 2.0), 1: (0.7, 0.7), 2: (1.8, 0.7)}

# save next to the loaded checkpoint
try:
    RUN_DIR = os.path.dirname(BEST_CHECKPOINT_PATH)
except NameError:
    RUN_DIR = "/content/drive/MyDrive/waymo_trajectory/run_20261004_2029"
PLOTS_DIR = os.path.join(RUN_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"Saving plots to {PLOTS_DIR}")

plot_paths = []
for _i in PLOT_SHARD_IDS:
    if not os.path.exists(_local(_i)):
        subprocess.run(["gsutil", "-m", "cp", _remote(_i), "/content/"], check=False)
    if os.path.exists(_local(_i)):
        plot_paths.append(_local(_i))
    else:
        print(f"shard {_i}: download failed -- re-run the auth cell, then this one")


def is_lane_change_like(scene_inputs, angle_threshold_deg=20):
    fut = scene_inputs["future_xy"].numpy()
    valid = scene_inputs["future_valid"].numpy().astype(bool)
    pts = fut[valid]
    if len(pts) < 12:
        return False
    v0 = pts[5] - pts[0]
    v1 = pts[-1] - pts[-6]
    if np.linalg.norm(v0) < 1e-3 or np.linalg.norm(v1) < 1e-3:
        return False
    cos_angle = np.dot(v0, v1) / (np.linalg.norm(v0) * np.linalg.norm(v1) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1, 1))) > angle_threshold_deg


def draw_faded_traj(ax, xy, color, lw=3.0, a_hi=0.85, a_lo=0.12, zorder=2):
    # thick line whose alpha fades over the horizon
    pts = xy.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    n = len(segs)
    rgba = np.zeros((n, 4))
    rgba[:, :3] = to_rgb(color)
    rgba[:, 3] = np.linspace(a_hi, a_lo, n)
    ax.add_collection(LineCollection(segs, colors=rgba, linewidths=lw, zorder=zorder, capstyle="round"))


def draw_car(ax, cx, cy, theta, length, width, facecolor, edgecolor, zorder, alpha=0.9):
    # oriented footprint with a pointed front
    l, w = length / 2.0, width / 2.0
    nose = length * 0.25
    local = np.array([[l, 0.0], [l - nose, w], [-l, w], [-l, -w], [l - nose, -w]])
    c, s = math.cos(theta), math.sin(theta)
    pts = local @ np.array([[c, -s], [s, c]]).T + np.array([cx, cy])
    ax.add_patch(Polygon(pts, closed=True, facecolor=facecolor, edgecolor=edgecolor,
                         linewidth=1.2, alpha=alpha, zorder=zorder))


def draw_agents(ax, agent_feats, agent_valid):
    # current-position footprints from agent-frame features (target = row 0)
    for i in range(agent_feats.shape[0]):
        vt = np.where(agent_valid[i] > 0)[0]
        if len(vt) == 0:
            continue
        t = vt[-1]
        px, py = agent_feats[i, t, 0] * POS_SCALE_M, agent_feats[i, t, 1] * POS_SCALE_M
        theta = math.atan2(agent_feats[i, t, 5], agent_feats[i, t, 4])
        oh = agent_feats[i, t, 6:9]
        ttype = int(np.argmax(oh)) if oh.sum() > 0 else -1
        length, width = CAR_SIZES.get(ttype, (2.0, 2.0))
        if i == 0:
            draw_car(ax, px, py, theta, length, width, "magenta", "darkmagenta", zorder=6)
        else:
            draw_car(ax, px, py, theta, length, width, "steelblue", "navy", zorder=5)


def plot_prediction(model, scene_inputs, viz_road_xy, viz_road_type, viz_road_id, title="Prediction"):
    model.eval()
    with torch.no_grad():
        batch = collate_scenes([scene_inputs])
        feats, valid, _, _, road_xy, road_type, road_valid, heading = [t.to(DEVICE) for t in batch]
        pred_trajs_t, pred_confs, _ = model(feats, valid, road_xy, road_type, road_valid, heading)
        sel_idx, _ = select_modes(pred_trajs_t, pred_confs, EVAL_TOP_K)
        order = sel_idx[0].cpu().numpy()
        probs = F.softmax(pred_confs, dim=-1)[0].cpu().numpy()
        pred_trajs = pred_trajs_t[0].cpu().numpy()
    model.train()

    agent_feats = scene_inputs["agent_feats"].float().numpy()   # [A, T, F], agent frame
    agent_valid = scene_inputs["agent_valid"].float().numpy()   # [A, T]
    gt = scene_inputs["future_xy"].numpy()
    gt_valid = scene_inputs["future_valid"].numpy().astype(bool)
    past = agent_feats[0, :, :2] * POS_SCALE_M
    past_valid = agent_valid[0] > 0
    M = pred_trajs.shape[0]

    # one square window framed on ground truth, history and the scored top-6
    framed = np.concatenate([pred_trajs[order].reshape(-1, 2), gt[gt_valid], past[past_valid], np.zeros((1, 2))])
    cx = 0.5 * (framed[:, 0].min() + framed[:, 0].max())
    cy = 0.5 * (framed[:, 1].min() + framed[:, 1].max())
    half = max(0.5 * max(np.ptp(framed[:, 0]), np.ptp(framed[:, 1])) + VIEW_MARGIN_M, VIEW_MIN_HALF_M)

    # (title, mode indices, show probability legend)
    panels = [("top-1", order[:1], True),
              (f"top-{len(order)} (scored)", order, True),
              (f"all {M} modes (far ones clipped)", np.arange(M), False)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
    cmap = plt.cm.viridis
    for ax, (ptitle, idxs, show_probs) in zip(axes, panels):
        draw_road_graph(ax, viz_road_xy, viz_road_type, viz_road_id)
        mode_handles = []
        for j, m in enumerate(idxs):
            color = cmap(j / max(len(idxs) - 1, 1))
            draw_faded_traj(ax, pred_trajs[m], color, lw=3.0)
            if show_probs:
                mode_handles.append(Line2D([0], [0], color=color, lw=3.0, label=f"mode {m}: p={probs[m]:.2f}"))
        # history, agent footprints, then ground truth on top
        ax.plot(past[past_valid, 0], past[past_valid, 1], "-", color="red", linewidth=2.0, zorder=3)
        draw_agents(ax, agent_feats, agent_valid)
        ax.plot(gt[gt_valid, 0], gt[gt_valid, 1], "-", color="black", linewidth=1.4, zorder=20)
        ax.set_xlim(cx - half, cx + half)
        ax.set_ylim(cy - half, cy + half)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(ptitle, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        if show_probs:
            ax.legend(handles=mode_handles, loc="upper right", fontsize=7, framealpha=0.85)

    handles = [
        Line2D([0], [0], marker="s", color="none", markerfacecolor="magenta",
               markeredgecolor="darkmagenta", markersize=11, label="target"),
        Line2D([0], [0], marker="s", color="none", markerfacecolor="steelblue",
               markeredgecolor="navy", markersize=10, label="other agents"),
        Line2D([0], [0], color="red", lw=2.0, label="history (1.1 s)"),
        Line2D([0], [0], color="black", lw=1.4, label="ground truth (future)"),
        Line2D([0], [0], color="0.25", lw=3.0, alpha=0.7, label="predicted modes (fade = time elapsed)"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False,
               bbox_to_anchor=(0.5, 0.015), fontsize=10)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])

    fname = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_") + ".png"
    fig.savefig(os.path.join(PLOTS_DIR, fname), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)


viz_candidates = []
for parsed_scene in load_dataset(plot_paths):
    samples = extract_scene_agents(parsed_scene, max_targets=1)
    if not samples:
        continue
    scene = samples[0]
    scene_inputs = build_model_inputs(scene)
    fv = scene_inputs["future_valid"].numpy() > 0
    if fv.sum() == 0 or np.linalg.norm(scene_inputs["future_xy"].numpy()[fv], axis=1).max() > MAX_PLAUSIBLE_COORD_M:
        continue
    wide_xy_raw, wide_type, wide_id, _ = extract_road_graph(
        parsed_scene, scene["origin"], radius=VIZ_ROAD_RADIUS_M, max_points=VIZ_MAX_ROAD_POINTS, square=True
    )
    wide_xy = to_agent_frame(wide_xy_raw, scene["origin"], scene["yaw0"])
    if len(scene["tl_xy"]):   # overlay traffic lights
        wide_xy = np.concatenate([wide_xy, to_agent_frame(scene["tl_xy"], scene["origin"], scene["yaw0"])])
        wide_type = np.concatenate([wide_type, scene["tl_type"]])
        wide_id = np.concatenate([wide_id, np.full(len(scene["tl_xy"]), -1, dtype=np.int64)])
    viz_candidates.append((scene_inputs, wide_xy, wide_type, wide_id))

lane_change_viz = [c for c in viz_candidates if is_lane_change_like(c[0])]
print(f"{len(viz_candidates)} TEST candidates, {len(lane_change_viz)} turn-like")
if not viz_candidates:
    raise RuntimeError("No scenes to plot -- the shard downloads failed. Re-run the auth cell, then this one.")

# Turn-like scenes
to_show = lane_change_viz[:N_VIZ_SAMPLES] if len(lane_change_viz) >= N_VIZ_SAMPLES else viz_candidates[:N_VIZ_SAMPLES]
for idx, (scene_inputs, wide_xy, wide_type, wide_id) in enumerate(to_show):
    plot_prediction(model, scene_inputs, wide_xy, wide_type, wide_id, title=f"TEST turn-like {idx:02d}")

# Random scenes (fixed seed)
for idx, (scene_inputs, wide_xy, wide_type, wide_id) in enumerate(
        random.Random(0).sample(viz_candidates, min(N_VIZ_RANDOM, len(viz_candidates)))):
    plot_prediction(model, scene_inputs, wide_xy, wide_type, wide_id, title=f"TEST random {idx:02d}")

# Worst scenes by top-6 minADE
cand_ade = compute_metrics(model, [c[0] for c in viz_candidates], return_per_sample=True)["per_sample"]["minADE"]
for rank, ci in enumerate(np.argsort(-cand_ade)[:N_VIZ_WORST]):
    scene_inputs, wide_xy, wide_type, wide_id = viz_candidates[ci]
    plot_prediction(model, scene_inputs, wide_xy, wide_type, wide_id,
                    title=f"TEST worst {rank:02d} (minADE {cand_ade[ci]:.1f} m)")


In [ ]:
# Top-6 vs perfect ranking (best of all 32 modes) on test, per agent type.

model.eval()
_names = ["Vehicle", "Pedestrian", "Cyclist"]
_acc = {k: [] for k in ["type", "ade6", "fde6", "ade32", "fde32"]}
print("model:", BEST_CHECKPOINT_PATH)
with torch.no_grad():
    for _i in range(0, len(test_scenes), 512):
        _b = test_scenes[_i:_i + 512]
        f, v, fx, fv, rx, rt, rv, h = [t.to(DEVICE) for t in collate_scenes(_b)]
        pt, pc, _ = model(f, v, rx, rt, rv, h)
        pt, pc = pt.float(), pc.float()
        B = len(_b)
        rows = torch.arange(B, device=DEVICE)
        dist = (pt - fx[:, None]).norm(dim=-1)                                   # [B, M, T]
        w = fv[:, None]
        ade = (dist * w).sum(-1) / w.sum(-1).clamp(min=1)                         # [B, M]
        last = (fv * torch.arange(fv.shape[1], device=DEVICE)).max(1).values.long()
        fde = dist[rows, :, last]                                                 # [B, M] at last valid step
        sel, _ = select_modes(pt, pc, EVAL_TOP_K)
        _acc["ade6"].append(torch.gather(ade, 1, sel).min(1).values.cpu())
        _acc["fde6"].append(torch.gather(fde, 1, sel).min(1).values.cpu())
        _acc["ade32"].append(ade.min(1).values.cpu())
        _acc["fde32"].append(fde.min(1).values.cpu())
        _acc["type"].append(torch.tensor([_target_type(s) for s in _b]))
_r = {k: torch.cat(v) for k, v in _acc.items()}


def _stats(m):
    return (int(m.sum()), _r["ade6"][m].mean().item(), _r["fde6"][m].mean().item(),
            (_r["fde6"][m] > 2.0).float().mean().item(), _r["ade32"][m].mean().item(),
            _r["fde32"][m].mean().item(), (_r["fde32"][m] > 2.0).float().mean().item())


_all = torch.ones(len(_r["type"]), dtype=torch.bool)
_rows = [("**All**", _stats(_all), True)] + [(n, _stats(_r["type"] == t), False) for t, n in enumerate(_names)]

print("\nCheck vs eval cell (All, top-6): minADE {:.2f}  minFDE {:.2f}  MR {:.1%}\n".format(*_rows[0][1][1:4]))
print("| Type | n | minADE (m) | minFDE (m) | Miss rate | minADE with perfect ranking (m) |")
print("|---|---|---|---|---|---|")
for name, s, bold in _rows:
    n, a6, f6, m6, a32, f32, m32 = s
    if bold:
        print(f"| {name} | {n:,} | **{a6:.2f}** | **{f6:.2f}** | **{m6:.1%}** | **{a32:.2f}** |")
    else:
        print(f"| {name} | {n:,} | {a6:.2f} | {f6:.2f} | {m6:.1%} | {a32:.2f} |")

print("\nFor reference, perfect ranking on the other metrics:")
for name, s, _ in _rows:
    print(f"  {name.strip('*'):<10} minFDE {s[5]:.2f} m  miss rate {s[6]:.1%}")

## Results

Held-out test split (60 shards, 131,465 samples), top-6 of 32 modes. Miss rate here is the fixed 2 m endpoint threshold at 8 s.

| Type | n | minADE (m) | minFDE (m) | Miss rate |
|---|---|---|---|---|
| **All** | 131,465 | **1.27** | **2.94** | **41.8%** |
| Vehicle | 113,960 | 1.36 | 3.14 | 45.1% |
| Pedestrian | 13,847 | 0.62 | 1.34 | 15.4% |
| Cyclist | 3,658 | 1.17 | 2.62 | 37.2% |

**Waymo-style metrics** (2 Hz points, speed-scaled lateral / longitudinal miss boxes):

| Horizon | minADE (m) | minFDE (m) | Miss rate |
|---|---|---|---|
| 3 s | 0.35 | 0.63 | 13.2% |
| 5 s | 0.71 | 1.42 | 17.0% |
| 8 s | 1.35 | 3.12 | 24.0% |
| **Average** | **0.80** | **1.72** | **18.1%** |

## Iterative Learnings/Notes:
- **Coverage is good.** Across the validation set, all 32 modes get used and are well spread out. The diversity is good.
- **Squiggles** Remaining 26 modes/trajectories and ot smooth, some modes are not getting trained. Initially the squiggles were a lot more with just 10 dataset shards, increasing to 600 helped a lot to smoothen out. Also adding mor road point into to the encder helped reduce the collisions rate.
- **Ranking was the weak spot.** The probabilites were flat in earlier iterations (0.05, 0.04 etc). The confidence head was getting mixed signals. The winner flickered between near identical mdoes. Grouping the maneuvers into differenr intents helped. As well as using a hybrid loss function that using Group + minADE(use endpoint distances before but ADE helped a lot) for Gaussian NLL and just Group for Confidence Cross Entropy. Ranking can still be improved. Maybe move to a different arch using Flow matching to get rid of the ranking and mode.


## Limitations

- Predicts one agent per forward pass, marginal, not joint over the scene.
- Does not predict heading/orientation.
- Trained on a subset of the dataset; the road graph uses point tokens rather than polyline encoders.

## References
1. [MULTIPATH++: EFFICIENT INFORMATION FUSION AND
TRAJECTORY AGGREGATION FOR BEHAVIOR PREDICTION
](https://arxiv.org/pdf/2111.14973)
2. [Wayformer: Motion Forecasting via Simple &
Efficient Attention Networks](https://arxiv.org/pdf/2207.05844)
3. [MTR-VP: Towards End-to-End Trajectory Planning through Context-Driven
Image Encoding and Multiple Trajectory Prediction](https://arxiv.org/pdf/2511.22181)
3. [VectorNet: Encoding HD Maps and Agent Dynamics from
Vectorized Representation](https://arxiv.org/pdf/2005.04259)
